In [ ]:
##### THIS NOTEBOOKS IS PHASE-1 STEP 3 - BGP DATA PROFILING

In [1]:
#verifying the environment
import sys

print(sys.executable)

/Users/taufiqtamin/opt/anaconda3/envs/bgp_ai/bin/python


In [2]:
#import the libraries needed
import pandas as pd
import pybgpkit_parser

print("Pandas:", pd.__version__)
print("pybgpkit-parser: OK")

Pandas: 3.0.5
pybgpkit-parser: OK


## 2. Dataset Overview

Load the processed BGP dataset and inspect its basic structure, including:

- Number of records
- Number of columns
- Column names
- Data types
- Memory usage

In [3]:
from pathlib import Path
import time

# Test dataset
DATA_FILE = Path("../data/processed_data/bgp_entries.csv.gz")

print("=" * 60)
print("BGP DATASET OVERVIEW")
print("=" * 60)
print(f"Input file : {DATA_FILE}")

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_FILE}")

print(f"File size  : {DATA_FILE.stat().st_size / (1024 * 1024):.2f} MB")
print()

BGP DATASET OVERVIEW
Input file : ../data/processed_data/bgp_entries.csv.gz
File size  : 0.50 MB



In [4]:
# Load the test dataset
start_time = time.time()

print("=" * 60)
print("LOADING BGP DATASET")
print("=" * 60)
print("Loading 100,000 test entries...")
print()

df = pd.read_csv(
    DATA_FILE,
    compression="gzip"
)

elapsed = time.time() - start_time

print("Dataset loaded successfully.")
print(f"Elapsed time : {elapsed:.2f} seconds")
print(f"Rows         : {len(df):,}")
print(f"Columns      : {len(df.columns)}")
print()

LOADING BGP DATASET
Loading 100,000 test entries...

Dataset loaded successfully.
Elapsed time : 0.25 seconds
Rows         : 100,000
Columns      : 9



## 2.1 Dataset Structure

Inspect the columns and data types of the processed BGP dataset.

In [5]:
print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)

print("\nColumns:")
for i, column in enumerate(df.columns, start=1):
    print(f"  {i}. {column}")

print("\nData types:")
print(df.dtypes)

print("\nShape:")
print(f"  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")

DATASET STRUCTURE

Columns:
  1. prefix
  2. peer_ip
  3. peer_asn
  4. next_hop
  5. as_path
  6. origin_asns
  7. origin
  8. local_pref
  9. med

Data types:
prefix           str
peer_ip          str
peer_asn       int64
next_hop         str
as_path          str
origin_asns      str
origin           str
local_pref     int64
med            int64
dtype: object

Shape:
  Rows    : 100,000
  Columns : 9


## 2.2 Data Quality Check

Check the dataset for:

- Missing values
- Duplicate records
- Empty strings
- Unexpected values in categorical fields

In [7]:
print("=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

# ----------------------------------------------------------
# 1. Missing values
# ----------------------------------------------------------
print("\nMissing values:")

missing = df.isna().sum()

for column, count in missing.items():
    print(f"  {column:15s}: {count:,}")

# ----------------------------------------------------------
# 2. Duplicate rows
# ----------------------------------------------------------
duplicate_count = df.duplicated().sum()

print(f"\nDuplicate rows : {duplicate_count:,}")

# ----------------------------------------------------------
# 3. Empty strings
# ----------------------------------------------------------
print("\nEmpty strings:")

string_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for column in string_columns:
    empty_count = (df[column].fillna("") == "").sum()
    print(f"  {column:15s}: {empty_count:,}")

# ----------------------------------------------------------
# 4. Unique values in categorical fields
# ----------------------------------------------------------
print("\nUnique values:")

for column in ["origin"]:
    print(f"  {column:15s}: {df[column].nunique():,}")

print("\nData quality check completed.")

DATA QUALITY CHECK

Missing values:
  prefix         : 0
  peer_ip        : 0
  peer_asn       : 0
  next_hop       : 0
  as_path        : 0
  origin_asns    : 0
  origin         : 0
  local_pref     : 0
  med            : 0

Duplicate rows : 0

Empty strings:
  prefix         : 0
  peer_ip        : 0
  next_hop       : 0
  as_path        : 0
  origin_asns    : 0
  origin         : 0

Unique values:
  origin         : 2

Data quality check completed.


## 2.4 Prefix Profiling

Analyze the BGP prefixes in the dataset, including:

- Number of unique prefixes
- Prefix length distribution
- IPv4 vs IPv6
- Most frequently observed prefixes

In [9]:
# ============================================================
# 2.4 Prefix Profiling
# ============================================================

import ipaddress
from collections import Counter

print("=" * 60)
print("PREFIX PROFILING")
print("=" * 60)

# ------------------------------------------------------------
# 1. Basic prefix statistics
# ------------------------------------------------------------

total_entries = len(df)
unique_prefixes = df["prefix"].nunique()

print(f"\nTotal BGP entries : {total_entries:,}")
print(f"Unique prefixes   : {unique_prefixes:,}")
#print(f"Duplicate entries : {total_entries - unique_prefixes:,}")
print(f"Additional entries beyond unique prefixes : {total_entries - unique_prefixes:,}")


# ------------------------------------------------------------
# 2. IPv4 / IPv6 classification
# ------------------------------------------------------------

def get_ip_version(prefix):
    try:
        return ipaddress.ip_network(prefix, strict=False).version
    except ValueError:
        return None


prefix_versions = df["prefix"].map(get_ip_version)

ipv4_count = (prefix_versions == 4).sum()
ipv6_count = (prefix_versions == 6).sum()
invalid_count = prefix_versions.isna().sum()

print("\nAddress family:")
print(f"IPv4 prefixes     : {ipv4_count:,}")
print(f"IPv6 prefixes     : {ipv6_count:,}")
print(f"Invalid prefixes  : {invalid_count:,}")


# ------------------------------------------------------------
# 3. Prefix-length distribution
# ------------------------------------------------------------

def get_prefix_length(prefix):
    try:
        return ipaddress.ip_network(prefix, strict=False).prefixlen
    except ValueError:
        return None


prefix_lengths = df["prefix"].map(get_prefix_length)

print("\nPrefix length distribution:")

length_distribution = (
    prefix_lengths
    .value_counts()
    .sort_index()
)

for prefix_length, count in length_distribution.items():
    print(f"  /{int(prefix_length):2d} : {count:,}")


# ------------------------------------------------------------
# 4. Most frequently observed prefixes
# ------------------------------------------------------------

print("\nTop 10 most frequently observed prefixes:")

top_prefixes = df["prefix"].value_counts().head(10)

for prefix, count in top_prefixes.items():
    print(f"  {prefix:20s} : {count:,}")


print("\nPrefix profiling completed.")

PREFIX PROFILING

Total BGP entries : 100,000
Unique prefixes   : 2,316
Additional entries beyond unique prefixes : 97,684

Address family:
IPv4 prefixes     : 100,000
IPv6 prefixes     : 0
Invalid prefixes  : 0

Prefix length distribution:
  / 0 : 11
  / 8 : 689
  / 9 : 618
  /10 : 1,693
  /11 : 4,227
  /12 : 13,068
  /13 : 25,566
  /14 : 51,720
  /15 : 2,408

Top 10 most frequently observed prefixes:
  34.112.0.0/14        : 51
  35.208.0.0/14        : 51
  35.212.0.0/14        : 51
  35.216.0.0/14        : 51
  136.32.0.0/11        : 50
  41.112.0.0/12        : 50
  105.208.0.0/12       : 50
  197.64.0.0/12        : 50
  105.208.0.0/13       : 50
  105.216.0.0/13       : 50

Prefix profiling completed.


## 2.5 Peer Profiling

Analyze the BGP peers contributing routes to the dataset, including:

- Number of unique peer IPs
- Number of unique peer ASNs
- Number of entries per peer
- Number of unique prefixes per peer
- Top peers by number of BGP entries

In [10]:
# ============================================================
# 2.5 Peer Profiling
# ============================================================

print("=" * 60)
print("PEER PROFILING")
print("=" * 60)

# ------------------------------------------------------------
# 1. Basic peer statistics
# ------------------------------------------------------------

unique_peers = df["peer_ip"].nunique()
unique_peer_asns = df["peer_asn"].nunique()

print(f"\nUnique peer IPs  : {unique_peers:,}")
print(f"Unique peer ASNs : {unique_peer_asns:,}")


# ------------------------------------------------------------
# 2. Entries per peer
# ------------------------------------------------------------

entries_per_peer = df["peer_ip"].value_counts()

print("\nTop 10 peers by number of BGP entries:")

for peer_ip, count in entries_per_peer.head(10).items():
    print(f"  {peer_ip:18s} : {count:,}")


# ------------------------------------------------------------
# 3. Unique prefixes per peer
# ------------------------------------------------------------

prefixes_per_peer = (
    df.groupby("peer_ip")["prefix"]
      .nunique()
      .sort_values(ascending=False)
)

print("\nTop 10 peers by number of unique prefixes:")

for peer_ip, count in prefixes_per_peer.head(10).items():
    print(f"  {peer_ip:18s} : {count:,}")


# ------------------------------------------------------------
# 4. Peer ASN distribution
# ------------------------------------------------------------

entries_per_asn = df["peer_asn"].value_counts()

print("\nTop 10 peer ASNs by number of BGP entries:")

for peer_asn, count in entries_per_asn.head(10).items():
    print(f"  AS{peer_asn:<12} : {count:,}")


print("\nPeer profiling completed.")

PEER PROFILING

Unique peer IPs  : 58
Unique peer ASNs : 52

Top 10 peers by number of BGP entries:
  49.12.70.222       : 2,307
  165.16.221.66      : 2,305
  89.234.186.6       : 2,303
  193.0.0.56         : 2,302
  45.134.89.1        : 2,301
  5.255.90.109       : 2,300
  193.163.86.231     : 2,300
  94.247.111.254     : 2,300
  5.178.95.254       : 2,299
  185.193.84.191     : 2,299

Top 10 peers by number of unique prefixes:
  49.12.70.222       : 2,307
  165.16.221.66      : 2,305
  89.234.186.6       : 2,303
  193.0.0.56         : 2,302
  45.134.89.1        : 2,301
  193.163.86.231     : 2,300
  5.255.90.109       : 2,300
  94.247.111.254     : 2,300
  5.178.95.254       : 2,299
  185.193.84.191     : 2,299

Top 10 peer ASNs by number of BGP entries:
  AS58057        : 4,659
  AS34927        : 4,588
  AS55720        : 4,586
  AS34800        : 2,621
  AS202365       : 2,548
  AS44393        : 2,307
  AS37721        : 2,305
  AS204092       : 2,303
  AS3333         : 2,302
  AS496

## 2.6 AS-Path Profiling

Analyze the AS paths in the test BGP dataset, including:

- AS-path length
- Minimum, maximum and average AS-path length
- AS-path length distribution
- Most frequently observed AS paths
- Potentially unusual long AS paths

In [11]:
# ============================================================
# 2.6 AS-Path Profiling
# ============================================================

print("=" * 60)
print("AS-PATH PROFILING")
print("=" * 60)


# ------------------------------------------------------------
# 1. Basic AS-path statistics
# ------------------------------------------------------------

total_entries = len(df)

empty_as_paths = (
    df["as_path"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(f"\nTotal BGP entries : {total_entries:,}")
print(f"Empty AS paths    : {empty_as_paths:,}")


# ------------------------------------------------------------
# 2. Calculate AS-path length
# ------------------------------------------------------------

def get_as_path_length(as_path):
    if not isinstance(as_path, str):
        return 0

    as_path = as_path.strip()

    if not as_path:
        return 0

    return len(as_path.split())


df["as_path_length"] = df["as_path"].map(get_as_path_length)


# ------------------------------------------------------------
# 3. AS-path statistics
# ------------------------------------------------------------

valid_paths = df.loc[
    df["as_path_length"] > 0,
    "as_path_length"
]

if len(valid_paths) > 0:

    print("\nAS-path length statistics:")

    print(f"  Minimum : {valid_paths.min():,}")
    print(f"  Maximum : {valid_paths.max():,}")
    print(f"  Average : {valid_paths.mean():.2f}")
    print(f"  Median  : {valid_paths.median():.0f}")


# ------------------------------------------------------------
# 4. AS-path length distribution
# ------------------------------------------------------------

print("\nAS-path length distribution:")

path_length_distribution = (
    df["as_path_length"]
    .value_counts()
    .sort_index()
)

for path_length, count in path_length_distribution.items():
    print(f"  Length {int(path_length):2d} : {count:,}")


# ------------------------------------------------------------
# 5. Most frequently observed AS paths
# ------------------------------------------------------------

print("\nTop 10 most frequently observed AS paths:")

top_as_paths = (
    df["as_path"]
    .value_counts()
    .head(10)
)

for as_path, count in top_as_paths.items():
    print(f"  {as_path:40s} : {count:,}")


# ------------------------------------------------------------
# 6. Long AS paths
# ------------------------------------------------------------

LONG_PATH_THRESHOLD = 10

long_paths = df[
    df["as_path_length"] >= LONG_PATH_THRESHOLD
]

print(
    f"\nAS paths with length >= "
    f"{LONG_PATH_THRESHOLD}: {len(long_paths):,}"
)


# ------------------------------------------------------------
# 7. Longest AS paths
# ------------------------------------------------------------

print("\nTop 10 longest AS paths:")

longest_paths = (
    df[["prefix", "peer_ip", "peer_asn", "as_path", "as_path_length"]]
    .sort_values("as_path_length", ascending=False)
    .head(10)
)

print(longest_paths.to_string(index=False))


print("\nAS-path profiling completed.")

AS-PATH PROFILING

Total BGP entries : 100,000
Empty AS paths    : 0

AS-path length statistics:
  Minimum : 1
  Maximum : 19
  Average : 4.31
  Median  : 4

AS-path length distribution:
  Length  1 : 102
  Length  2 : 5,550
  Length  3 : 27,792
  Length  4 : 31,783
  Length  5 : 17,438
  Length  6 : 9,152
  Length  7 : 4,708
  Length  8 : 1,773
  Length  9 : 507
  Length 10 : 389
  Length 11 : 256
  Length 12 : 90
  Length 13 : 30
  Length 14 : 58
  Length 15 : 29
  Length 16 : 61
  Length 17 : 154
  Length 18 : 112
  Length 19 : 16

Top 10 most frequently observed AS paths:
  55720 23764 4134                         : 178
  34549 3257 4134                          : 144
  37721 174 4134                           : 144
  202365 50673 174 4134                    : 144
  22652 1299 4134                          : 144
  44393 35710 4809 4134                    : 144
  204092 20766 1299 4134                   : 144
  34800 58057 50673 174 4134               : 144
  45049 6696 1299 4134   

## 2.7 BGP Attribute Profiling

Profile the main BGP path-selection attributes:

- ORIGIN
- LOCAL_PREF
- MED

In [12]:
# ============================================================
# BGP ATTRIBUTE PROFILING
# ============================================================

print("=" * 60)
print("BGP ATTRIBUTE PROFILING")
print("=" * 60)

print(f"\nTotal BGP entries : {len(df):,}")

# ------------------------------------------------------------
# 1. ORIGIN attribute
# ------------------------------------------------------------

print("\nORIGIN attribute distribution:")

origin_counts = df["origin"].value_counts(dropna=False)

for origin, count in origin_counts.items():
    print(f"  {str(origin):10s} : {count:,}")

print(f"Unique ORIGIN values : {df['origin'].nunique(dropna=False):,}")


# ------------------------------------------------------------
# 2. LOCAL_PREF
# ------------------------------------------------------------

print("\nLOCAL_PREF statistics:")

print(f"  Minimum : {df['local_pref'].min():,}")
print(f"  Maximum : {df['local_pref'].max():,}")
print(f"  Average : {df['local_pref'].mean():.2f}")
print(f"  Median  : {df['local_pref'].median():.2f}")

print("\nTop 10 LOCAL_PREF values:")

local_pref_counts = df["local_pref"].value_counts().head(10)

for value, count in local_pref_counts.items():
    print(f"  {value:10,} : {count:,}")


# ------------------------------------------------------------
# 3. MED
# ------------------------------------------------------------

print("\nMED statistics:")

print(f"  Minimum : {df['med'].min():,}")
print(f"  Maximum : {df['med'].max():,}")
print(f"  Average : {df['med'].mean():.2f}")
print(f"  Median  : {df['med'].median():.2f}")

print("\nTop 10 MED values:")

med_counts = df["med"].value_counts().head(10)

for value, count in med_counts.items():
    print(f"  {value:10,} : {count:,}")


print("\nBGP attribute profiling completed.")

BGP ATTRIBUTE PROFILING

Total BGP entries : 100,000

ORIGIN attribute distribution:
  IGP        : 96,582
  INCOMPLETE : 3,418
Unique ORIGIN values : 2

LOCAL_PREF statistics:
  Minimum : 0
  Maximum : 0
  Average : 0.00
  Median  : 0.00

Top 10 LOCAL_PREF values:
           0 : 100,000

MED statistics:
  Minimum : 0
  Maximum : 18,811
  Average : 40.60
  Median  : 0.00

Top 10 MED values:
           0 : 92,272
         150 : 2,303
          50 : 2,299
       1,000 : 2,291
         100 : 684
      18,811 : 49
       1,430 : 32
       5,645 : 31
       1,730 : 6
       3,219 : 6

BGP attribute profiling completed.


## 6. Next-Hop Profiling

Analyze the `next_hop` field, including:

- Number of unique next hops
- IPv4 vs IPv6 next hops
- Invalid next-hop values
- Most frequently observed next hops
- Next-hop distribution
- Most common prefixes associated with each next hop

In [13]:
# ============================================================
# NEXT-HOP PROFILING
# ============================================================

import ipaddress

print("=" * 60)
print("NEXT-HOP PROFILING")
print("=" * 60)

total_entries = len(df)

# ------------------------------------------------------------
# 1. Basic statistics
# ------------------------------------------------------------

unique_next_hops = df["next_hop"].nunique()

print(f"\nTotal BGP entries : {total_entries:,}")
print(f"Unique next hops  : {unique_next_hops:,}")


# ------------------------------------------------------------
# 2. Validate next-hop addresses
# ------------------------------------------------------------

def classify_ip(value):
    try:
        ip = ipaddress.ip_address(str(value).strip())

        if ip.version == 4:
            return "IPv4"
        elif ip.version == 6:
            return "IPv6"

    except ValueError:
        return "Invalid"

    return "Invalid"


next_hop_types = df["next_hop"].apply(classify_ip)

print("\nAddress family:")

print(f"IPv4 next hops    : {(next_hop_types == 'IPv4').sum():,}")
print(f"IPv6 next hops    : {(next_hop_types == 'IPv6').sum():,}")
print(f"Invalid next hops : {(next_hop_types == 'Invalid').sum():,}")


# ------------------------------------------------------------
# 3. Most frequently observed next hops
# ------------------------------------------------------------

print("\nTop 10 most frequently observed next hops:")

top_next_hops = df["next_hop"].value_counts().head(10)

for next_hop, count in top_next_hops.items():
    print(f"  {next_hop:15} : {count:,}")


# ------------------------------------------------------------
# 4. Most common prefixes associated with top next hops
# ------------------------------------------------------------

print("\nTop prefixes associated with the 5 most common next hops:")

top_5_next_hops = top_next_hops.head(5).index

for next_hop in top_5_next_hops:

    prefixes = (
        df[df["next_hop"] == next_hop]["prefix"]
        .value_counts()
        .head(5)
    )

    print(f"\nNext hop: {next_hop}")

    for prefix, count in prefixes.items():
        print(f"  {prefix:20} : {count:,}")


print("\nNext-hop profiling completed.")

NEXT-HOP PROFILING

Total BGP entries : 100,000
Unique next hops  : 59

Address family:
IPv4 next hops    : 100,000
IPv6 next hops    : 0
Invalid next hops : 0

Top 10 most frequently observed next hops:
  49.12.70.222    : 2,307
  165.16.221.66   : 2,305
  89.234.186.6    : 2,303
  193.0.0.56      : 2,302
  45.134.89.1     : 2,301
  5.255.90.109    : 2,300
  193.163.86.231  : 2,300
  94.247.111.254  : 2,300
  5.178.95.254    : 2,299
  185.193.84.191  : 2,299

Top prefixes associated with the 5 most common next hops:

Next hop: 49.12.70.222
  0.0.0.0/0            : 1
  7.0.0.0/8            : 1
  11.0.0.0/8           : 1
  12.0.0.0/8           : 1
  17.0.0.0/8           : 1

Next hop: 165.16.221.66
  7.0.0.0/8            : 1
  11.0.0.0/8           : 1
  12.0.0.0/8           : 1
  17.0.0.0/8           : 1
  21.0.0.0/8           : 1

Next hop: 89.234.186.6
  7.0.0.0/8            : 1
  11.0.0.0/8           : 1
  12.0.0.0/8           : 1
  17.0.0.0/8           : 1
  21.0.0.0/8           : 1

In [15]:
print(df.shape)

(100000, 10)


In [16]:
print(df.columns.tolist())

['prefix', 'peer_ip', 'peer_asn', 'next_hop', 'as_path', 'origin_asns', 'origin', 'local_pref', 'med', 'as_path_length']


In [17]:
COLLECTOR_ASN = 12654

collector_peer_count = (
    df["peer_asn"] == COLLECTOR_ASN
).sum()

print("=" * 60)
print("COLLECTOR / PEER ASN CHECK")
print("=" * 60)

print(f"Collector ASN           : AS{COLLECTOR_ASN}")
print(f"Entries from AS{COLLECTOR_ASN}: {collector_peer_count:,}")

if collector_peer_count == 0:
    print("\nNo peer entries from the collector ASN.")
else:
    print("\nWARNING: Collector ASN appears in peer_asn.")

COLLECTOR / PEER ASN CHECK
Collector ASN           : AS12654
Entries from AS12654: 0

No peer entries from the collector ASN.


## 2.6 MRT Collector / Viewpoint Profiling

Identify the MRT collector/viewpoint information and understand
the relationship between the collector and observed BGP peers.

In [18]:
# Inspect MRT header / parser metadata

import pybgpkit_parser

RAW_FILE = "data/raw_data/latest-bview.gz"

parser = pybgpkit_parser.Parser(RAW_FILE)

print("=" * 60)
print("MRT / BGP VIEWPOINT INSPECTION")
print("=" * 60)

print(f"Input file : {RAW_FILE}")
print()

print("Parser attributes:")
print([attr for attr in dir(parser) if not attr.startswith("_")])

ValueError: Error: IO error: No such file or directory (os error 2)

In [20]:
# Inspect MRT header / parser metadata

import pybgpkit_parser

RAW_FILE = "../data/raw_data/latest-bview.gz"

parser = pybgpkit_parser.Parser(RAW_FILE)

print("=" * 60)
print("MRT / BGP VIEWPOINT INSPECTION")
print("=" * 60)

print(f"Input file : {RAW_FILE}")
print()

print("Parser attributes:")
print([attr for attr in dir(parser) if not attr.startswith("_")])

MRT / BGP VIEWPOINT INSPECTION
Input file : ../data/raw_data/latest-bview.gz

Parser attributes:
['count', 'from_filters', 'iter_batches', 'iter_tuple_batches', 'iter_tuples', 'parse_all', 'parse_next']


### Note on Local / Collector ASN

The `peer_asn` field identifies the ASN of the BGP peer
observed by the MRT collector.

The collector's own ASN cannot be inferred simply by
looking for a particular ASN in `peer_asn`.

Therefore, we do not hard-code a local/collector ASN
in this profiling stage.

## 2.7 Peer Relationship Profiling

Analyze the relationship between observed BGP peer IP addresses
and their peer ASNs.

Checks include:
- Number of unique peer IPs
- Number of unique peer ASNs
- Peer IP → ASN mapping
- ASNs represented by multiple peer IPs
- Peer IPs associated with multiple ASNs
- Number of BGP entries contributed by each peer

In [21]:
# Peer IP / ASN relationship profiling

print("=" * 60)
print("PEER RELATIONSHIP PROFILING")
print("=" * 60)

# Basic counts
unique_peer_ips = df["peer_ip"].nunique()
unique_peer_asns = df["peer_asn"].nunique()

print(f"Unique peer IPs  : {unique_peer_ips:,}")
print(f"Unique peer ASNs : {unique_peer_asns:,}")
print()

# ------------------------------------------------------------
# 1. Peer IP -> ASN mapping
# ------------------------------------------------------------

peer_ip_asn = (
    df.groupby("peer_ip")["peer_asn"]
      .nunique()
)

multi_asn_peer_ips = peer_ip_asn[peer_ip_asn > 1]

print("Peer IPs associated with multiple ASNs:")
print(f"Count : {len(multi_asn_peer_ips):,}")
print()

# ------------------------------------------------------------
# 2. ASN -> number of peer IPs
# ------------------------------------------------------------

asn_peer_ip_count = (
    df.groupby("peer_asn")["peer_ip"]
      .nunique()
      .sort_values(ascending=False)
)

print("Top 10 ASNs by number of peer IPs:")

for asn, count in asn_peer_ip_count.head(10).items():
    print(f"  AS{asn:<10} : {count:,}")

print()

# ------------------------------------------------------------
# 3. Top peers by number of BGP entries
# ------------------------------------------------------------

peer_entries = (
    df.groupby(["peer_ip", "peer_asn"])
      .size()
      .sort_values(ascending=False)
)

print("Top 10 peer IP / ASN combinations by BGP entries:")

for (peer_ip, peer_asn), count in peer_entries.head(10).items():
    print(
        f"  {peer_ip:<16} AS{peer_asn:<8} : {count:,}"
    )

print()

# ------------------------------------------------------------
# 4. Check whether one peer IP maps to multiple ASNs
# ------------------------------------------------------------

if len(multi_asn_peer_ips) == 0:
    print("Peer IP consistency check : PASS")
    print("Every peer IP maps to exactly one peer ASN.")
else:
    print("Peer IP consistency check : WARNING")
    print("Some peer IPs are associated with multiple ASNs.")

    print("\nExamples:")
    for peer_ip in multi_asn_peer_ips.head(10).index:
        asns = sorted(
            df.loc[df["peer_ip"] == peer_ip, "peer_asn"].unique()
        )
        print(f"  {peer_ip} -> {asns}")

print()
print("Peer relationship profiling completed.")

PEER RELATIONSHIP PROFILING
Unique peer IPs  : 58
Unique peer ASNs : 52

Peer IPs associated with multiple ASNs:
Count : 0

Top 10 ASNs by number of peer IPs:
  AS58057      : 3
  AS55720      : 2
  AS202365     : 2
  AS34927      : 2
  AS34800      : 2
  AS852        : 1
  AS59919      : 1
  AS49673      : 1
  AS50300      : 1
  AS50304      : 1

Top 10 peer IP / ASN combinations by BGP entries:
  49.12.70.222     AS44393    : 2,307
  165.16.221.66    AS37721    : 2,305
  89.234.186.6     AS204092   : 2,303
  193.0.0.56       AS3333     : 2,302
  45.134.89.1      AS34927    : 2,301
  193.163.86.231   AS34800    : 2,300
  5.255.90.109     AS202365   : 2,300
  94.247.111.254   AS49673    : 2,300
  5.178.95.254     AS59919    : 2,299
  185.193.84.191   AS29504    : 2,299

Peer IP consistency check : PASS
Every peer IP maps to exactly one peer ASN.

Peer relationship profiling completed.


## 2.8 Prefix Multipath / Route Candidate Profiling

Analyze how many BGP route observations exist for each prefix.

A prefix may be advertised by multiple BGP peers. These observations
represent route candidates and must not be treated as erroneous
duplicates.

This stage measures route diversity only. It does not determine
the BGP best path.

In [22]:
# Prefix multipath / route candidate profiling

print("=" * 60)
print("PREFIX MULTIPATH / ROUTE CANDIDATE PROFILING")
print("=" * 60)

total_entries = len(df)
unique_prefixes = df["prefix"].nunique()

# Number of route observations for each prefix
prefix_route_counts = (
    df.groupby("prefix")
      .size()
      .sort_values(ascending=False)
)

print(f"Total BGP entries      : {total_entries:,}")
print(f"Unique prefixes        : {unique_prefixes:,}")
print()

# ------------------------------------------------------------
# Distribution of route candidates per prefix
# ------------------------------------------------------------

candidate_distribution = (
    prefix_route_counts
    .value_counts()
    .sort_index()
)

print("Route candidate distribution:")
print()

for candidate_count, prefix_count in candidate_distribution.items():
    print(
        f"  {candidate_count:>3} route candidate(s) : "
        f"{prefix_count:,} prefix(es)"
    )

# ------------------------------------------------------------
# Basic statistics
# ------------------------------------------------------------

print()
print("Route candidate statistics:")

print(f"  Minimum : {prefix_route_counts.min():,}")
print(f"  Maximum : {prefix_route_counts.max():,}")
print(f"  Average : {prefix_route_counts.mean():.2f}")
print(f"  Median  : {prefix_route_counts.median():.0f}")

# ------------------------------------------------------------
# Multipath prefixes
# ------------------------------------------------------------

multipath_prefixes = prefix_route_counts[
    prefix_route_counts > 1
]

single_path_prefixes = prefix_route_counts[
    prefix_route_counts == 1
]

print()
print("Prefix classification:")

print(
    f"  Single-candidate prefixes : "
    f"{len(single_path_prefixes):,}"
)

print(
    f"  Multi-candidate prefixes  : "
    f"{len(multipath_prefixes):,}"
)

# ------------------------------------------------------------
# Top prefixes by number of route candidates
# ------------------------------------------------------------

print()
print("Top 10 prefixes by number of route candidates:")

for prefix, count in prefix_route_counts.head(10).items():
    print(f"  {prefix:<20} : {count:,}")

print()
print("Prefix multipath profiling completed.")

PREFIX MULTIPATH / ROUTE CANDIDATE PROFILING
Total BGP entries      : 100,000
Unique prefixes        : 2,316

Route candidate distribution:

    1 route candidate(s) : 7 prefix(es)
    2 route candidate(s) : 2 prefix(es)
    3 route candidate(s) : 1 prefix(es)
   10 route candidate(s) : 5 prefix(es)
   11 route candidate(s) : 1 prefix(es)
   13 route candidate(s) : 4 prefix(es)
   15 route candidate(s) : 1 prefix(es)
   17 route candidate(s) : 2 prefix(es)
   18 route candidate(s) : 1 prefix(es)
   31 route candidate(s) : 6 prefix(es)
   35 route candidate(s) : 3 prefix(es)
   37 route candidate(s) : 1 prefix(es)
   41 route candidate(s) : 25 prefix(es)
   42 route candidate(s) : 516 prefix(es)
   43 route candidate(s) : 887 prefix(es)
   44 route candidate(s) : 422 prefix(es)
   45 route candidate(s) : 175 prefix(es)
   46 route candidate(s) : 109 prefix(es)
   47 route candidate(s) : 7 prefix(es)
   48 route candidate(s) : 77 prefix(es)
   49 route candidate(s) : 46 prefix(es)
   50 

## 2.9 Prefix Route Diversity Profiling

A BGP prefix may be observed from multiple peers and ASNs.

This profiling measures the diversity of route candidates available for
each prefix before any BGP best-path selection is applied.

We will examine:

- Number of unique peer IPs
- Number of unique peer ASNs
- Number of unique AS paths
- Number of unique next-hops
- Prefixes with the highest route diversity

This analysis describes the observed BGP route candidates. It does not
attempt to determine the best path selected by a router.

In [23]:
# ============================================================
# #2.9 PREFIX / PEER / ASN RELATIONSHIP PROFILING
# ============================================================

print("=" * 60)
print("PREFIX / PEER / ASN RELATIONSHIP PROFILING")
print("=" * 60)

# Number of unique peers and ASNs per prefix
prefix_relationship = (
    df.groupby("prefix")
      .agg(
          route_candidates=("prefix", "size"),
          unique_peers=("peer_ip", "nunique"),
          unique_peer_asns=("peer_asn", "nunique"),
      )
      .sort_values("route_candidates", ascending=False)
)

print(f"Total unique prefixes : {len(prefix_relationship):,}")
print()

# ------------------------------------------------------------
# 1. Distribution of unique peers per prefix
# ------------------------------------------------------------

print("Unique peers per prefix:")
print(prefix_relationship["unique_peers"].describe().to_string())
print()

# ------------------------------------------------------------
# 2. Distribution of unique ASNs per prefix
# ------------------------------------------------------------

print("Unique peer ASNs per prefix:")
print(prefix_relationship["unique_peer_asns"].describe().to_string())
print()

# ------------------------------------------------------------
# 3. Check whether route candidates == unique peers
# ------------------------------------------------------------

multiple_peer_observation = (
    prefix_relationship["route_candidates"]
    != prefix_relationship["unique_peers"]
).sum()

print(
    f"Prefixes where route candidates != unique peers : "
    f"{multiple_peer_observation:,}"
)
print()

# ------------------------------------------------------------
# 4. Prefixes with the highest number of peers
# ------------------------------------------------------------

print("Top 10 prefixes by unique peer count:")

top_peer_prefixes = (
    prefix_relationship
    .sort_values(
        ["unique_peers", "route_candidates"],
        ascending=False
    )
    .head(10)
)

for prefix, row in top_peer_prefixes.iterrows():
    print(
        f"  {prefix:<18} : "
        f"{int(row['unique_peers']):>2} peers, "
        f"{int(row['unique_peer_asns']):>2} ASNs, "
        f"{int(row['route_candidates']):>2} candidates"
    )

print()

# ------------------------------------------------------------
# 5. Prefixes observed from multiple ASNs
# ------------------------------------------------------------

multi_asn_prefixes = (
    prefix_relationship["unique_peer_asns"] > 1
).sum()

print(
    f"Prefixes observed from multiple peer ASNs : "
    f"{multi_asn_prefixes:,}"
)

# ------------------------------------------------------------
# 6. Maximum relationship example
# ------------------------------------------------------------

max_peer_count = prefix_relationship["unique_peers"].max()

print()
print(
    f"Maximum unique peers observed for one prefix : "
    f"{max_peer_count}"
)

example_prefixes = prefix_relationship[
    prefix_relationship["unique_peers"] == max_peer_count
].head(5)

print()
print("Example prefixes with maximum peer count:")

for prefix, row in example_prefixes.iterrows():
    print(
        f"  {prefix:<18} : "
        f"{int(row['unique_peers'])} peers, "
        f"{int(row['unique_peer_asns'])} ASNs"
    )

print()
print("Prefix / peer / ASN relationship profiling completed.")

PREFIX / PEER / ASN RELATIONSHIP PROFILING
Total unique prefixes : 2,316

Unique peers per prefix:
count    2316.000000
mean       43.177893
std         4.078745
min         1.000000
25%        43.000000
50%        43.000000
75%        44.000000
max        51.000000

Unique peer ASNs per prefix:
count    2316.000000
mean       39.939983
std         3.683403
min         1.000000
25%        39.000000
50%        40.000000
75%        41.000000
max        46.000000

Prefixes where route candidates != unique peers : 0

Top 10 prefixes by unique peer count:
  35.216.0.0/14      : 51 peers, 46 ASNs, 51 candidates
  35.212.0.0/14      : 51 peers, 46 ASNs, 51 candidates
  34.112.0.0/14      : 51 peers, 46 ASNs, 51 candidates
  35.208.0.0/14      : 51 peers, 46 ASNs, 51 candidates
  105.216.0.0/13     : 50 peers, 45 ASNs, 50 candidates
  180.72.0.0/14      : 50 peers, 45 ASNs, 50 candidates
  41.112.0.0/12      : 50 peers, 45 ASNs, 50 candidates
  136.32.0.0/11      : 50 peers, 45 ASNs, 50 candid

In [24]:
# ============================================================
# 2.10 PREFIX ROUTE DIVERSITY PROFILING
# ============================================================

print("=" * 60)
print("PREFIX ROUTE DIVERSITY PROFILING")
print("=" * 60)

# ------------------------------------------------------------
# Calculate route diversity for each prefix
# ------------------------------------------------------------

prefix_diversity = (
    df.groupby("prefix")
      .agg(
          route_candidates=("prefix", "size"),
          unique_peers=("peer_ip", "nunique"),
          unique_peer_asns=("peer_asn", "nunique"),
          unique_as_paths=("as_path", "nunique"),
          unique_next_hops=("next_hop", "nunique"),
      )
      .sort_values("route_candidates", ascending=False)
)

print(f"Total unique prefixes : {len(prefix_diversity):,}")
print()

# ------------------------------------------------------------
# Diversity statistics
# ------------------------------------------------------------

print("Route diversity statistics:")

for column in [
    "route_candidates",
    "unique_peers",
    "unique_peer_asns",
    "unique_as_paths",
    "unique_next_hops",
]:
    print(f"\n{column}:")
    print(prefix_diversity[column].describe().to_string())

# ------------------------------------------------------------
# Check whether each route candidate has a unique peer
# ------------------------------------------------------------

candidate_peer_difference = (
    prefix_diversity["route_candidates"]
    != prefix_diversity["unique_peers"]
).sum()

print()
print(
    f"Prefixes where route candidates != unique peers : "
    f"{candidate_peer_difference:,}"
)

# ------------------------------------------------------------
# Top prefixes by AS-path diversity
# ------------------------------------------------------------

print()
print("Top 10 prefixes by unique AS-path count:")

top_as_path_diversity = (
    prefix_diversity
    .sort_values(
        ["unique_as_paths", "route_candidates"],
        ascending=False
    )
    .head(10)
)

for prefix, row in top_as_path_diversity.iterrows():
    print(
        f"  {prefix:<18} : "
        f"{int(row['unique_as_paths']):>2} AS paths, "
        f"{int(row['unique_peers']):>2} peers, "
        f"{int(row['unique_peer_asns']):>2} ASNs, "
        f"{int(row['unique_next_hops']):>2} next-hops"
    )

# ------------------------------------------------------------
# Top prefixes by next-hop diversity
# ------------------------------------------------------------

print()
print("Top 10 prefixes by unique next-hop count:")

top_next_hop_diversity = (
    prefix_diversity
    .sort_values(
        ["unique_next_hops", "route_candidates"],
        ascending=False
    )
    .head(10)
)

for prefix, row in top_next_hop_diversity.iterrows():
    print(
        f"  {prefix:<18} : "
        f"{int(row['unique_next_hops']):>2} next-hops, "
        f"{int(row['unique_peers']):>2} peers, "
        f"{int(row['unique_as_paths']):>2} AS paths"
    )

# ------------------------------------------------------------
# Prefixes with identical AS paths from multiple peers
# ------------------------------------------------------------

same_path_multiple_peers = (
    df.groupby(["prefix", "as_path"])["peer_ip"]
      .nunique()
)

repeated_as_path_groups = (
    same_path_multiple_peers > 1
).sum()

print()
print(
    f"Prefix / AS-path combinations observed from "
    f"multiple peers : {repeated_as_path_groups:,}"
)

print()
print("Prefix route diversity profiling completed.")

PREFIX ROUTE DIVERSITY PROFILING
Total unique prefixes : 2,316

Route diversity statistics:

route_candidates:
count    2316.000000
mean       43.177893
std         4.078745
min         1.000000
25%        43.000000
50%        43.000000
75%        44.000000
max        51.000000

unique_peers:
count    2316.000000
mean       43.177893
std         4.078745
min         1.000000
25%        43.000000
50%        43.000000
75%        44.000000
max        51.000000

unique_peer_asns:
count    2316.000000
mean       39.939983
std         3.683403
min         1.000000
25%        39.000000
50%        40.000000
75%        41.000000
max        46.000000

unique_as_paths:
count    2316.000000
mean       42.909758
std         3.942138
min         1.000000
25%        42.000000
50%        43.000000
75%        44.000000
max        50.000000

unique_next_hops:
count    2316.000000
mean       43.177893
std         4.078745
min         1.000000
25%        43.000000
50%        43.000000
75%        44.000000

## 2.11 AS-Path Structural / Anomaly Profiling

AS-path length was examined earlier as part of basic AS-path profiling.

This section goes deeper into the structure and consistency of AS paths.

We will examine:

- AS-path loops and repeated ASNs
- AS-path prepending
- Origin ASN consistency
- AS-path and origin field consistency
- Very long or unusual AS paths
- Potentially malformed AS-path structures

The objective is to establish the normal structural characteristics of
AS paths and identify potential data-quality or routing anomalies before
processing the full dataset.

In [30]:
# ============================================================
# AS-PATH STRUCTURAL / ANOMALY PROFILING
# ============================================================

print("=" * 60)
print("AS-PATH STRUCTURAL / ANOMALY PROFILING")
print("=" * 60)

total_entries = len(df)

print(f"Total BGP entries : {total_entries:,}")
print()

# ------------------------------------------------------------
# 1. Parse AS paths
# ------------------------------------------------------------

def parse_as_path(path):
    if pd.isna(path):
        return []
    
    return str(path).split()


df["as_path_list"] = df["as_path"].apply(parse_as_path)

# ------------------------------------------------------------
# 2. AS-path loops / repeated ASNs
# ------------------------------------------------------------

def has_repeated_asn(as_path):
    return len(as_path) != len(set(as_path))


df["as_path_has_repeat"] = df["as_path_list"].apply(has_repeated_asn)

repeat_count = df["as_path_has_repeat"].sum()

print("AS-path repeated ASN analysis:")
print(f"Entries with repeated ASN : {repeat_count:,}")
print(
    f"Percentage                : "
    f"{repeat_count / total_entries * 100:.2f}%"
)
print()

# ------------------------------------------------------------
# 3. AS-path prepending
# ------------------------------------------------------------

def has_prepending(as_path):
    if len(as_path) < 2:
        return False
    
    return any(
        as_path[i] == as_path[i + 1]
        for i in range(len(as_path) - 1)
    )


df["as_path_has_prepend"] = df["as_path_list"].apply(has_prepending)

prepend_count = df["as_path_has_prepend"].sum()

print("AS-path prepending:")
print(f"Entries with prepending : {prepend_count:,}")
print(
    f"Percentage              : "
    f"{prepend_count / total_entries * 100:.2f}%"
)
print()

# ------------------------------------------------------------
# 4. Origin ASN consistency
# ------------------------------------------------------------

def get_origin_asns(value):
    """
    Normalize origin_asns into a list of ASN strings.

    Handles values such as:
        [174]
        ['174']
        174
        '174'
    """

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return [str(asn) for asn in value]

    value = str(value).strip()

    # Handle string representation such as:
    # "[174, 64500]"
    if value.startswith("[") and value.endswith("]"):
        value = value[1:-1]

    value = value.replace(",", " ")

    return [
        item.strip().strip("'\"")
        for item in value.split()
        if item.strip()
    ]


df["origin_asns_normalized"] = df["origin_asns"].apply(
    get_origin_asns
)


def get_last_asns_from_path(as_path):
    """
    Extract the ASN(s) represented by the final AS-path element.

    Normal ASN:
        6877

    AS_SET:
        {6877}

    AS_SET with multiple ASNs:
        {6877, 64500}
    """

    if not as_path:
        return []

    last_element = str(as_path[-1]).strip()

    # Normal ASN
    if not (
        last_element.startswith("{")
        and last_element.endswith("}")
    ):
        return [last_element]

    # AS_SET
    content = last_element[1:-1]

    return [
        asn.strip()
        for asn in content.replace(",", " ").split()
        if asn.strip()
    ]


df["as_path_last_asns"] = df["as_path_list"].apply(
    get_last_asns_from_path
)


def origin_matches_as_path(row):
    """
    Origin ASN is considered consistent when at least one
    origin ASN is represented by the final AS-path element.

    This correctly handles both:
        ... 6877
        ... {6877}
    """

    origin_list = set(row["origin_asns_normalized"])
    last_asns = set(row["as_path_last_asns"])

    if not origin_list or not last_asns:
        return False

    return bool(origin_list.intersection(last_asns))


df["origin_as_path_match"] = df.apply(
    origin_matches_as_path,
    axis=1
)


origin_mismatch_count = (
    ~df["origin_as_path_match"]
).sum()


print("Origin ASN consistency:")
print(
    f"Entries with origin mismatch : "
    f"{origin_mismatch_count:,}"
)

print(
    f"Percentage                  : "
    f"{origin_mismatch_count / total_entries * 100:.2f}%"
)

print()
# ------------------------------------------------------------
# 5. Very long AS paths
# ------------------------------------------------------------

LONG_PATH_THRESHOLD = 10

long_path_count = (
    df["as_path_length"] >= LONG_PATH_THRESHOLD
).sum()

print("Long AS paths:")
print(
    f"Paths with length >= {LONG_PATH_THRESHOLD} : "
    f"{long_path_count:,}"
)
print(
    f"Percentage                       : "
    f"{long_path_count / total_entries * 100:.2f}%"
)
print()

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print("=" * 60)
print("AS-PATH STRUCTURAL / ANOMALY SUMMARY")
print("=" * 60)

print(f"Total entries                 : {total_entries:,}")
print(f"Repeated ASN entries          : {repeat_count:,}")
print(f"Prepending entries            : {prepend_count:,}")
print(f"Origin ASN mismatches         : {origin_mismatch_count:,}")
print(f"Long AS paths (>= 10)         : {long_path_count:,}")

print()
print("AS-path structural profiling completed.")

AS-PATH STRUCTURAL / ANOMALY PROFILING
Total BGP entries : 100,000

AS-path repeated ASN analysis:
Entries with repeated ASN : 17,419
Percentage                : 17.42%

AS-path prepending:
Entries with prepending : 17,417
Percentage              : 17.42%

Origin ASN consistency:
Entries with origin mismatch : 0
Percentage                  : 0.00%

Long AS paths:
Paths with length >= 10 : 1,195
Percentage                       : 1.20%

AS-PATH STRUCTURAL / ANOMALY SUMMARY
Total entries                 : 100,000
Repeated ASN entries          : 17,419
Prepending entries            : 17,417
Origin ASN mismatches         : 0
Long AS paths (>= 10)         : 1,195

AS-path structural profiling completed.


### 2.11.1 Consecutive vs. Non-Consecutive ASN Repetition

Repeated ASNs in an AS-path do not necessarily indicate a routing anomaly.

Consecutive repetitions are normally associated with AS-path prepending, where an AS intentionally repeats its ASN to influence BGP path selection.

Non-consecutive repetitions are structurally different and may indicate an AS-path loop, unusual routing behaviour, or malformed data.

This subsection separates the two cases.

We will examine:

- AS-paths containing consecutive ASN repetitions
- AS-paths containing non-consecutive ASN repetitions
- AS-paths containing both
- Examples of non-consecutive repetitions for further investigation

In [28]:
# ============================================================
# 2.11.1 CONSECUTIVE VS. NON-CONSECUTIVE ASN REPETITION
# ============================================================

print("=" * 60)
print("CONSECUTIVE VS. NON-CONSECUTIVE ASN REPETITION")
print("=" * 60)


# ------------------------------------------------------------
# Detect consecutive ASN repetition
# ------------------------------------------------------------

def has_consecutive_repeat(as_path):
    if len(as_path) < 2:
        return False

    return any(
        as_path[i] == as_path[i + 1]
        for i in range(len(as_path) - 1)
    )


# ------------------------------------------------------------
# Remove consecutive duplicates
#
# Example:
# A A A B C C D
#
# becomes:
# A B C D
#
# This allows us to distinguish AS-path prepending
# from genuine non-consecutive repetition.
# ------------------------------------------------------------

def collapse_consecutive_asns(as_path):
    if not as_path:
        return []

    collapsed = [as_path[0]]

    for asn in as_path[1:]:
        if asn != collapsed[-1]:
            collapsed.append(asn)

    return collapsed


# ------------------------------------------------------------
# Detect genuine non-consecutive repetition
#
# Example:
#
# A A A B C C
# -> A B C
# -> no non-consecutive repetition
#
# A B A C
# -> A B A C
# -> A is repeated non-consecutively
# ------------------------------------------------------------

def has_non_consecutive_repeat(as_path):

    collapsed = collapse_consecutive_asns(as_path)

    return len(collapsed) != len(set(collapsed))


# ------------------------------------------------------------
# Apply classification
# ------------------------------------------------------------

df["has_consecutive_repeat"] = df["as_path_list"].apply(
    has_consecutive_repeat
)

df["has_non_consecutive_repeat"] = df["as_path_list"].apply(
    has_non_consecutive_repeat
)


# ------------------------------------------------------------
# Counts
# ------------------------------------------------------------

consecutive_count = (
    df["has_consecutive_repeat"].sum()
)

non_consecutive_count = (
    df["has_non_consecutive_repeat"].sum()
)

both_count = (
    df["has_consecutive_repeat"]
    & df["has_non_consecutive_repeat"]
).sum()


print()
print("ASN repetition classification:")

print(
    f"Consecutive repetition     : "
    f"{consecutive_count:,}"
)

print(
    f"Non-consecutive repetition : "
    f"{non_consecutive_count:,}"
)

print(
    f"Both types                 : "
    f"{both_count:,}"
)


# ------------------------------------------------------------
# Percentages
# ------------------------------------------------------------

print()
print("Percentages:")

print(
    f"Consecutive repetition     : "
    f"{consecutive_count / total_entries * 100:.2f}%"
)

print(
    f"Non-consecutive repetition : "
    f"{non_consecutive_count / total_entries * 100:.2f}%"
)


# ------------------------------------------------------------
# Show genuine non-consecutive examples
# ------------------------------------------------------------

print()
print("Examples of genuine non-consecutive ASN repetition:")

non_consecutive_examples = df[
    df["has_non_consecutive_repeat"]
][
    [
        "prefix",
        "peer_ip",
        "peer_asn",
        "as_path",
        "as_path_length"
    ]
].head(10)


if len(non_consecutive_examples) == 0:
    print("None found.")
else:
    print(
        non_consecutive_examples.to_string(
            index=False
        )
    )


print()
print(
    "Consecutive / non-consecutive ASN analysis completed."
)

CONSECUTIVE VS. NON-CONSECUTIVE ASN REPETITION

ASN repetition classification:
Consecutive repetition     : 17,417
Non-consecutive repetition : 2
Both types                 : 0

Percentages:
Consecutive repetition     : 17.42%
Non-consecutive repetition : 0.00%

Examples of genuine non-consecutive ASN repetition:
        prefix       peer_ip  peer_asn                   as_path  as_path_length
154.152.0.0/13 102.217.156.3    328977 328977 36926 327708 36926               4
 154.76.0.0/14 102.217.156.3    328977 328977 36926 327708 36926               4

Consecutive / non-consecutive ASN analysis completed.


### 2.11.2 Origin ASN Mismatch Investigation

The previous profiling identified a small number of entries where the
origin ASN reported in the `origin_asns` field does not match the final
ASN in the AS-path.

This subsection investigates those records in detail.

We will examine:

- Number of mismatched entries
- Peer ASN
- Origin ASN reported by the dataset
- Final ASN in the AS-path
- Complete AS-path
- Prefix
- AS-path length
- Whether the mismatch follows a common pattern

The purpose is investigation only. No records will be modified or removed.

In [32]:
# ============================================================
# 2.11.2 ORIGIN ASN MISMATCH INVESTIGATION
# ============================================================

print("=" * 60)
print("ORIGIN ASN MISMATCH INVESTIGATION")
print("=" * 60)

# ------------------------------------------------------------
# Identify mismatch records
# ------------------------------------------------------------

origin_mismatches = df[
    ~df["origin_as_path_match"]
].copy()

print()
print(
    f"Total origin ASN mismatches : "
    f"{len(origin_mismatches):,}"
)

print(
    f"Percentage of dataset       : "
    f"{len(origin_mismatches) / total_entries * 100:.2f}%"
)

# ------------------------------------------------------------
# Prepare investigation columns
# ------------------------------------------------------------

origin_mismatches["as_path_last_asns"] = (
    origin_mismatches["as_path_list"]
    .apply(lambda x: x[-1] if x else None)
)

origin_mismatches["origin_asns_display"] = (
    origin_mismatches["origin_asns_normalized"]
    .apply(lambda x: ",".join(x))
)

# ------------------------------------------------------------
# Display all mismatches
# ------------------------------------------------------------

print()
print("Origin ASN mismatch records:")
print()

mismatch_view = origin_mismatches[
    [
        "prefix",
        "peer_ip",
        "peer_asn",
        "origin_asns_display",
        "as_path_last_asns",
        "as_path",
        "as_path_length"
    ]
]

print(
    mismatch_view.to_string(index=False)
)

# ------------------------------------------------------------
# Compare peer ASN and AS-path final ASN
# ------------------------------------------------------------

print()
print("Mismatch relationship analysis:")
print()

peer_equals_last = (
    origin_mismatches["peer_asn"].astype(str)
    ==
    origin_mismatches["as_path_last_asns"].astype(str)
).sum()

peer_equals_origin = (
    origin_mismatches["peer_asn"].astype(str)
    ==
    origin_mismatches["origin_asns_display"].astype(str)
).sum()

print(
    f"Peer ASN = final AS-path ASN : "
    f"{peer_equals_last:,}"
)

print(
    f"Peer ASN = origin ASN        : "
    f"{peer_equals_origin:,}"
)

# ------------------------------------------------------------
# Origin ASN mismatch patterns
# ------------------------------------------------------------

print()
print("Most common origin ASN values in mismatches:")

print(
    origin_mismatches[
        "origin_asns_display"
    ]
    .value_counts()
    .head(10)
    .to_string()
)

print()
print("Most common final AS-path ASNs in mismatches:")

print(
    origin_mismatches[
        "as_path_last_asn"
    ]
    .value_counts()
    .head(10)
    .to_string()
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("ORIGIN ASN MISMATCH INVESTIGATION SUMMARY")
print("=" * 60)

print(
    f"Total mismatches : "
    f"{len(origin_mismatches):,}"
)

print(
    f"Mismatch rate    : "
    f"{len(origin_mismatches) / total_entries * 100:.2f}%"
)

print()
print("No records were modified or removed.")
print(
    "Origin ASN mismatch investigation completed."
)

ORIGIN ASN MISMATCH INVESTIGATION

Total origin ASN mismatches : 0
Percentage of dataset       : 0.00%

Origin ASN mismatch records:

Empty DataFrame
Columns: [prefix, peer_ip, peer_asn, origin_asns_display, as_path_last_asns, as_path, as_path_length]
Index: []

Mismatch relationship analysis:

Peer ASN = final AS-path ASN : 0
Peer ASN = origin ASN        : 0

Most common origin ASN values in mismatches:
Series([], )

Most common final AS-path ASNs in mismatches:
Series([], )

ORIGIN ASN MISMATCH INVESTIGATION SUMMARY
Total mismatches : 0
Mismatch rate    : 0.00%

No records were modified or removed.
Origin ASN mismatch investigation completed.


## 2.11.3 AS-Path Prepending Investigation

In [33]:
# ============================================================
# 2.11.3 AS-PATH PREPENDING INVESTIGATION
# ============================================================

print("=" * 60)
print("AS-PATH PREPENDING INVESTIGATION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Identify entries with AS-path prepending
# ------------------------------------------------------------

prepending_entries = df[
    df["as_path_has_prepend"]
].copy()

print()
print(
    f"Total BGP entries        : "
    f"{len(df):,}"
)

print(
    f"Entries with prepending  : "
    f"{len(prepending_entries):,}"
)

print(
    f"Prepending percentage    : "
    f"{len(prepending_entries) / total_entries * 100:.2f}%"
)

# ------------------------------------------------------------
# 2. Count maximum consecutive repetition
# ------------------------------------------------------------

def max_consecutive_repeat(as_path):
    """
    Return the maximum number of consecutive occurrences
    of the same ASN in an AS path.

    Examples:
        3356 2152 2152 2152 7377
        -> 3

        3356 2152 2152 7377
        -> 2

        3356 2152 7377
        -> 1
    """

    if not as_path:
        return 0

    max_repeat = 1
    current_repeat = 1

    for i in range(1, len(as_path)):

        if as_path[i] == as_path[i - 1]:
            current_repeat += 1
            max_repeat = max(
                max_repeat,
                current_repeat
            )
        else:
            current_repeat = 1

    return max_repeat


prepending_entries["max_consecutive_repeat"] = (
    prepending_entries["as_path_list"]
    .apply(max_consecutive_repeat)
)

# ------------------------------------------------------------
# 3. Distribution of prepend depth
# ------------------------------------------------------------

print()
print("Prepending depth distribution:")

print(
    prepending_entries[
        "max_consecutive_repeat"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# 4. Most frequently prepended ASNs
# ------------------------------------------------------------

def get_prepended_asns(as_path):
    """
    Return ASNs that occur consecutively at least twice.
    """

    prepended = []

    if len(as_path) < 2:
        return prepended

    for i in range(len(as_path) - 1):

        if as_path[i] == as_path[i + 1]:

            if not prepended or prepended[-1] != as_path[i]:
                prepended.append(as_path[i])

    return prepended


prepending_entries["prepended_asns"] = (
    prepending_entries["as_path_list"]
    .apply(get_prepended_asns)
)

all_prepended_asns = []

for asns in prepending_entries["prepended_asns"]:
    all_prepended_asns.extend(asns)

print()
print("Top 10 most frequently prepended ASNs:")

print(
    pd.Series(all_prepended_asns)
    .value_counts()
    .head(10)
    .to_string()
)

# ------------------------------------------------------------
# 5. Most heavily prepended AS paths
# ------------------------------------------------------------

print()
print("Top 10 entries by maximum prepend depth:")

prepend_view = prepending_entries[
    [
        "prefix",
        "peer_ip",
        "peer_asn",
        "as_path",
        "as_path_length",
        "max_consecutive_repeat"
    ]
].sort_values(
    "max_consecutive_repeat",
    ascending=False
)

print(
    prepend_view
    .head(10)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 6. Extreme prepending cases
# ------------------------------------------------------------

EXTREME_PREPEND_THRESHOLD = 5

extreme_prepending = prepending_entries[
    prepending_entries["max_consecutive_repeat"]
    >= EXTREME_PREPEND_THRESHOLD
]

print()
print(
    f"Entries with prepend depth >= "
    f"{EXTREME_PREPEND_THRESHOLD} : "
    f"{len(extreme_prepending):,}"
)

# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("AS-PATH PREPENDING INVESTIGATION SUMMARY")
print("=" * 60)

print(
    f"Total entries              : "
    f"{len(df):,}"
)

print(
    f"Entries with prepending    : "
    f"{len(prepending_entries):,}"
)

print(
    f"Prepending rate            : "
    f"{len(prepending_entries) / total_entries * 100:.2f}%"
)

print(
    f"Maximum prepend depth      : "
    f"{prepending_entries['max_consecutive_repeat'].max()}"
)

print(
    f"Extreme prepending (>=5)   : "
    f"{len(extreme_prepending):,}"
)

print()
print(
    "AS-path prepending investigation completed."
)

AS-PATH PREPENDING INVESTIGATION

Total BGP entries        : 100,000
Entries with prepending  : 17,417
Prepending percentage    : 17.42%

Prepending depth distribution:
max_consecutive_repeat
2     4737
3     2551
4     7186
5     1560
6      737
7       19
8      176
9        4
10       2
11     107
13     338

Top 10 most frequently prepended ASNs:
16509    4100
721      1886
4538     1470
9318     1307
4766      818
26615     817
4134      502
6128      391
36992     339
12322     330

Top 10 entries by maximum prepend depth:
        prefix        peer_ip  peer_asn                                                                                                       as_path  as_path_length  max_consecutive_repeat
156.184.0.0/13  31.169.49.228     50304            50304 1299 3257 8452 36992 36992 36992 36992 36992 36992 36992 36992 36992 36992 36992 36992 36992              17                      13
156.184.0.0/13 185.102.84.232     61218       61218 6939 1299 3257 8452 36992 36992 3

## 2.11.4 Extreme AS-Path Prepending Investigation

Very deep AS-path prepending may be legitimate routing policy, but it
deserves closer investigation.

This section examines entries with a prepend depth of 5 or greater.

We will examine:

- ASNs responsible for extreme prepending
- Actual prepend counts per ASN
- Peers associated with extreme prepending
- Prefixes affected
- Distribution of extreme prepend depths
- The most extreme AS-paths

The objective is to distinguish legitimate routing policy from
potentially unusual or malformed AS-path data.

In [35]:
# ============================================================
# 2.11.4 EXTREME AS-PATH PREPENDING INVESTIGATION
# ============================================================

print("=" * 60)
print("EXTREME AS-PATH PREPENDING INVESTIGATION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Calculate maximum consecutive ASN repetition
# ------------------------------------------------------------

def max_consecutive_repeat(as_path):
    """
    Return the maximum number of consecutive occurrences
    of the same ASN in an AS-path.

    Example:
        3356 36992 36992 36992 7377

    Result:
        3
    """

    if not as_path:
        return 0

    max_repeat = 1
    current_repeat = 1

    for i in range(1, len(as_path)):

        if as_path[i] == as_path[i - 1]:
            current_repeat += 1
        else:
            current_repeat = 1

        if current_repeat > max_repeat:
            max_repeat = current_repeat

    return max_repeat


# Create the column explicitly
df["max_consecutive_repeat"] = (
    df["as_path_list"]
    .apply(max_consecutive_repeat)
)

# ------------------------------------------------------------
# 2. Select extreme prepending entries
# ------------------------------------------------------------

EXTREME_PREPEND_THRESHOLD = 5

extreme_prepending = df[
    df["max_consecutive_repeat"] >= EXTREME_PREPEND_THRESHOLD
].copy()

print()
print(f"Total BGP entries             : {len(df):,}")
print(
    f"Extreme prepending entries    : "
    f"{len(extreme_prepending):,}"
)
print(
    f"Extreme prepending percentage : "
    f"{len(extreme_prepending) / total_entries * 100:.2f}%"
)

# ------------------------------------------------------------
# 3. Distribution of extreme prepend depths
# ------------------------------------------------------------

print()
print("Extreme prepend depth distribution:")

print(
    extreme_prepending[
        "max_consecutive_repeat"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# 4. Identify the actual ASN being prepended
# ------------------------------------------------------------

def get_prepend_runs(as_path):
    """
    Return consecutive ASN repetition runs.

    Example:
        3356 36992 36992 36992 7377

    Result:
        [('36992', 3)]
    """

    runs = []

    if not as_path:
        return runs

    current_asn = as_path[0]
    current_count = 1

    for i in range(1, len(as_path)):

        if as_path[i] == current_asn:
            current_count += 1

        else:
            if current_count >= 2:
                runs.append(
                    (current_asn, current_count)
                )

            current_asn = as_path[i]
            current_count = 1

    # Handle final run
    if current_count >= 2:
        runs.append(
            (current_asn, current_count)
        )

    return runs


extreme_prepending["prepend_runs"] = (
    extreme_prepending["as_path_list"]
    .apply(get_prepend_runs)
)

# ------------------------------------------------------------
# 5. Build table of extreme prepend runs
# ------------------------------------------------------------

prepend_run_records = []

for _, row in extreme_prepending.iterrows():

    for asn, count in row["prepend_runs"]:

        if count >= EXTREME_PREPEND_THRESHOLD:

            prepend_run_records.append(
                {
                    "asn": asn,
                    "prepend_count": count,
                    "prefix": row["prefix"],
                    "peer_ip": row["peer_ip"],
                    "peer_asn": row["peer_asn"],
                    "as_path": row["as_path"]
                }
            )


prepend_runs_df = pd.DataFrame(
    prepend_run_records
)

# ------------------------------------------------------------
# 6. ASN frequency
# ------------------------------------------------------------

print()
print("Top 10 ASNs by extreme prepend occurrences:")

if not prepend_runs_df.empty:

    print(
        prepend_runs_df["asn"]
        .value_counts()
        .head(10)
        .to_string()
    )

else:

    print("No extreme prepend runs found.")

# ------------------------------------------------------------
# 7. Total prepend depth by ASN
# ------------------------------------------------------------

print()
print("Top 10 ASNs by total extreme prepend depth:")

if not prepend_runs_df.empty:

    print(
        prepend_runs_df
        .groupby("asn")["prepend_count"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .to_string()
    )

else:

    print("No extreme prepend runs found.")

# ------------------------------------------------------------
# 8. Peers associated with extreme prepending
# ------------------------------------------------------------

print()
print("Top 10 peers associated with extreme prepending:")

if not extreme_prepending.empty:

    print(
        extreme_prepending["peer_ip"]
        .value_counts()
        .head(10)
        .to_string()
    )

else:

    print("No extreme prepending entries found.")

# ------------------------------------------------------------
# 9. Peer ASNs associated with extreme prepending
# ------------------------------------------------------------

print()
print("Top 10 peer ASNs associated with extreme prepending:")

if not extreme_prepending.empty:

    print(
        extreme_prepending["peer_asn"]
        .value_counts()
        .head(10)
        .to_string()
    )

else:

    print("No extreme prepending entries found.")

# ------------------------------------------------------------
# 10. Prefixes affected
# ------------------------------------------------------------

print()
print("Top 10 prefixes affected by extreme prepending:")

if not extreme_prepending.empty:

    print(
        extreme_prepending["prefix"]
        .value_counts()
        .head(10)
        .to_string()
    )

else:

    print("No extreme prepending entries found.")

# ------------------------------------------------------------
# 11. Most extreme AS-paths
# ------------------------------------------------------------

print()
print("Top 10 most extreme AS paths:")

extreme_view = extreme_prepending[
    [
        "prefix",
        "peer_ip",
        "peer_asn",
        "as_path",
        "as_path_length",
        "max_consecutive_repeat"
    ]
].sort_values(
    "max_consecutive_repeat",
    ascending=False
)

print(
    extreme_view
    .head(10)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 12. Maximum prepend depth
# ------------------------------------------------------------

maximum_prepend_depth = (
    extreme_prepending[
        "max_consecutive_repeat"
    ].max()
)

print()
print(
    f"Maximum prepend depth : "
    f"{maximum_prepend_depth}"
)

# ------------------------------------------------------------
# 13. Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("EXTREME AS-PATH PREPENDING INVESTIGATION SUMMARY")
print("=" * 60)

print(
    f"Extreme threshold          : "
    f">= {EXTREME_PREPEND_THRESHOLD}"
)

print(
    f"Extreme prepending entries : "
    f"{len(extreme_prepending):,}"
)

print(
    f"Percentage of dataset      : "
    f"{len(extreme_prepending) / total_entries * 100:.2f}%"
)

print(
    f"Maximum prepend depth      : "
    f"{maximum_prepend_depth}"
)

print()
print(
    "Extreme AS-path prepending investigation completed."
)

EXTREME AS-PATH PREPENDING INVESTIGATION

Total BGP entries             : 100,000
Extreme prepending entries    : 2,943
Extreme prepending percentage : 2.94%

Extreme prepend depth distribution:
max_consecutive_repeat
5     1560
6      737
7       19
8      176
9        4
10       2
11     107
13     338

Top 10 ASNs by extreme prepend occurrences:
asn
9318     1307
36992     338
6306      176
37069     172
4134      161
11830     132
4637      126
12479      96
13285      91
9121       55

Top 10 ASNs by total extreme prepend depth:
asn
9318     6535
36992    4394
6306     1056
37069     990
4134      805
11830     792
12479     768
4637      756
13285     695
29465     528

Top 10 peers associated with extreme prepending:
peer_ip
198.58.198.252    238
102.208.105.2     100
194.28.97.2        93
193.150.22.240     87
31.169.49.228      85
12.0.1.63          82
45.61.0.85         78
193.160.39.1       78
208.51.134.248     77
80.77.16.114       76

Top 10 peer ASNs associated with extr

## 2.11.5 Extreme Prepending Pattern / Impact Analysis

In [36]:
# ============================================================
# 2.11.5 EXTREME PREPENDING PATTERN / IMPACT ANALYSIS
# ============================================================

print("=" * 60)
print("EXTREME PREPENDING PATTERN / IMPACT ANALYSIS")
print("=" * 60)

EXTREME_PREPEND_THRESHOLD = 5

# ------------------------------------------------------------
# Identify extreme-prepending entries
# ------------------------------------------------------------

extreme_prepending = df[
    df["max_consecutive_repeat"] >= EXTREME_PREPEND_THRESHOLD
].copy()

print()
print(f"Total BGP entries              : {len(df):,}")
print(
    f"Extreme prepending entries     : "
    f"{len(extreme_prepending):,}"
)
print(
    f"Extreme prepending percentage  : "
    f"{len(extreme_prepending) / len(df) * 100:.2f}%"
)

# ------------------------------------------------------------
# 1. ASN impact
# ------------------------------------------------------------

print()
print("Top ASNs by affected prefix count:")

asn_prefix_impact = (
    extreme_prepending
    .explode("as_path_list")
    .groupby("as_path_list")["prefix"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

print(asn_prefix_impact.to_string())

# ------------------------------------------------------------
# 2. ASN impact by peer count
# ------------------------------------------------------------

print()
print("Top ASNs by affected peer count:")

asn_peer_impact = (
    extreme_prepending
    .explode("as_path_list")
    .groupby("as_path_list")["peer_ip"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

print(asn_peer_impact.to_string())

# ------------------------------------------------------------
# 3. Maximum prepend depth by ASN
# ------------------------------------------------------------

print()
print("Top ASNs by maximum prepend depth:")

asn_max_depth = (
    extreme_prepending
    .explode("as_path_list")
    .groupby("as_path_list")["max_consecutive_repeat"]
    .max()
    .sort_values(ascending=False)
    .head(10)
)

print(asn_max_depth.to_string())

# ------------------------------------------------------------
# 4. Prefix impact
# ------------------------------------------------------------

print()
print("Top prefixes by number of extreme-prepend observations:")

prefix_impact = (
    extreme_prepending
    .groupby("prefix")
    .agg(
        extreme_entries=("prefix", "size"),
        unique_peers=("peer_ip", "nunique"),
        max_prepend_depth=("max_consecutive_repeat", "max"),
    )
    .sort_values(
        ["extreme_entries", "unique_peers"],
        ascending=False
    )
    .head(10)
)

print(prefix_impact.to_string())

# ------------------------------------------------------------
# 5. Peer impact
# ------------------------------------------------------------

print()
print("Top peers by number of extreme-prepend observations:")

peer_impact = (
    extreme_prepending
    .groupby("peer_ip")
    .agg(
        extreme_entries=("peer_ip", "size"),
        unique_prefixes=("prefix", "nunique"),
        max_prepend_depth=("max_consecutive_repeat", "max"),
    )
    .sort_values(
        ["extreme_entries", "unique_prefixes"],
        ascending=False
    )
    .head(10)
)

print(peer_impact.to_string())

# ------------------------------------------------------------
# 6. Cross-peer propagation
# ------------------------------------------------------------

print()
print("Prefixes observed with extreme prepending from multiple peers:")

multi_peer_extreme = (
    extreme_prepending
    .groupby("prefix")
    .agg(
        unique_peers=("peer_ip", "nunique"),
        unique_peer_asns=("peer_asn", "nunique"),
        max_prepend_depth=("max_consecutive_repeat", "max"),
    )
    .query("unique_peers > 1")
    .sort_values(
        ["unique_peers", "max_prepend_depth"],
        ascending=False
    )
)

print(
    f"Prefixes with extreme prepending from "
    f"multiple peers : {len(multi_peer_extreme):,}"
)

print()
print(multi_peer_extreme.head(10).to_string())

# ------------------------------------------------------------
# 7. Prepend depth vs AS-path length
# ------------------------------------------------------------

print()
print("Prepend depth vs AS-path length:")

depth_impact = (
    extreme_prepending
    .groupby("max_consecutive_repeat")
    .agg(
        entries=("prefix", "size"),
        avg_as_path_length=("as_path_length", "mean"),
        max_as_path_length=("as_path_length", "max"),
        unique_prefixes=("prefix", "nunique"),
    )
    .sort_index()
)

print(depth_impact.to_string())

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("EXTREME PREPENDING PATTERN / IMPACT SUMMARY")
print("=" * 60)

print(
    f"Extreme threshold              : "
    f">= {EXTREME_PREPEND_THRESHOLD}"
)

print(
    f"Extreme prepending entries     : "
    f"{len(extreme_prepending):,}"
)

print(
    f"Unique affected prefixes       : "
    f"{extreme_prepending['prefix'].nunique():,}"
)

print(
    f"Unique affected peers          : "
    f"{extreme_prepending['peer_ip'].nunique():,}"
)

print(
    f"Unique affected peer ASNs      : "
    f"{extreme_prepending['peer_asn'].nunique():,}"
)

print(
    f"Maximum prepend depth          : "
    f"{extreme_prepending['max_consecutive_repeat'].max()}"
)

print(
    f"Prefixes seen from multiple "
    f"peers                         : "
    f"{len(multi_peer_extreme):,}"
)

print()
print(
    "Extreme prepending pattern / "
    "impact analysis completed."
)

EXTREME PREPENDING PATTERN / IMPACT ANALYSIS

Total BGP entries              : 100,000
Extreme prepending entries     : 2,943
Extreme prepending percentage  : 2.94%

Top ASNs by affected prefix count:
as_path_list
1403      238
577       200
701       196
4134      160
327727    100
174       100
328840    100
45049      93
6696       92
1299       91

Top ASNs by affected peer count:
as_path_list
29465    48
16637    48
10361    44
6306     44
24863    44
3356     44
37069    44
24691    44
19978    44
8452     44

Top ASNs by maximum prepend depth:
as_path_list
53356    13
577      13
3257     13
58057    13
6939     13
7018     13
29504    13
44393    13
45049    13
29169    13

Top prefixes by number of extreme-prepend observations:
                extreme_entries  unique_peers  max_prepend_depth
prefix                                                          
102.88.0.0/13                48            48                 11
105.180.0.0/14               44            44             

## 2.11.6 Actual Prepending ASN Attribution

In [37]:
# ============================================================
# 2.11.6 ACTUAL PREPENDING ASN ATTRIBUTION
# ============================================================

print("=" * 60)
print("ACTUAL PREPENDING ASN ATTRIBUTION")
print("=" * 60)

# ------------------------------------------------------------
# Find the ASN responsible for the maximum consecutive repeat
# ------------------------------------------------------------

def get_max_repeated_asn(as_path):
    if not as_path:
        return None, 0

    max_asn = None
    max_count = 1

    current_asn = as_path[0]
    current_count = 1

    for asn in as_path[1:]:
        if asn == current_asn:
            current_count += 1
        else:
            if current_count > max_count:
                max_asn = current_asn
                max_count = current_count

            current_asn = asn
            current_count = 1

    # Check final run
    if current_count > max_count:
        max_asn = current_asn
        max_count = current_count

    return max_asn, max_count


extreme_prepending[
    ["actual_prepend_asn", "actual_prepend_depth"]
] = extreme_prepending["as_path_list"].apply(
    lambda x: pd.Series(get_max_repeated_asn(x))
)

# ------------------------------------------------------------
# Verify attribution
# ------------------------------------------------------------

verification = (
    extreme_prepending["actual_prepend_depth"]
    ==
    extreme_prepending["max_consecutive_repeat"]
).all()

print()
print(
    f"ASN attribution verification : "
    f"{'PASS' if verification else 'FAIL'}"
)

# ------------------------------------------------------------
# Top ASNs by extreme prepend occurrences
# ------------------------------------------------------------

print()
print("Top ASNs by actual extreme prepend occurrences:")

asn_occurrences = (
    extreme_prepending[
        extreme_prepending["actual_prepend_asn"].notna()
    ]
    ["actual_prepend_asn"]
    .value_counts()
    .head(10)
)

print(asn_occurrences.to_string())

# ------------------------------------------------------------
# Top ASNs by total prepend depth
# ------------------------------------------------------------

print()
print("Top ASNs by total extreme prepend depth:")

asn_total_depth = (
    extreme_prepending
    .groupby("actual_prepend_asn")
    ["actual_prepend_depth"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(asn_total_depth.to_string())

# ------------------------------------------------------------
# Top ASNs by affected prefixes
# ------------------------------------------------------------

print()
print("Top ASNs by affected prefixes:")

asn_prefixes = (
    extreme_prepending
    .groupby("actual_prepend_asn")
    ["prefix"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

print(asn_prefixes.to_string())

# ------------------------------------------------------------
# Top ASNs by affected peers
# ------------------------------------------------------------

print()
print("Top ASNs by affected peers:")

asn_peers = (
    extreme_prepending
    .groupby("actual_prepend_asn")
    ["peer_ip"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

print(asn_peers.to_string())

# ------------------------------------------------------------
# Maximum prepend depth by actual ASN
# ------------------------------------------------------------

print()
print("Maximum prepend depth by actual ASN:")

asn_max_depth = (
    extreme_prepending
    .groupby("actual_prepend_asn")
    ["actual_prepend_depth"]
    .max()
    .sort_values(ascending=False)
    .head(10)
)

print(asn_max_depth.to_string())

# ------------------------------------------------------------
# Most extreme actual prepend examples
# ------------------------------------------------------------

print()
print("Top 10 most extreme actual prepending examples:")

extreme_examples = (
    extreme_prepending[
        [
            "prefix",
            "peer_ip",
            "peer_asn",
            "actual_prepend_asn",
            "actual_prepend_depth",
            "as_path",
            "as_path_length"
        ]
    ]
    .sort_values(
        "actual_prepend_depth",
        ascending=False
    )
    .head(10)
)

print(
    extreme_examples.to_string(index=False)
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("ACTUAL PREPENDING ASN ATTRIBUTION SUMMARY")
print("=" * 60)

print(
    f"Extreme entries analyzed : "
    f"{len(extreme_prepending):,}"
)

print(
    f"Unique actual prepending ASNs : "
    f"{extreme_prepending['actual_prepend_asn'].nunique():,}"
)

print(
    f"Maximum prepend depth : "
    f"{extreme_prepending['actual_prepend_depth'].max()}"
)

print()
print(
    "Actual prepending ASN attribution completed."
)

ACTUAL PREPENDING ASN ATTRIBUTION

ASN attribution verification : PASS

Top ASNs by actual extreme prepend occurrences:
actual_prepend_asn
9318     1307
36992     338
6306      176
37069     172
4134      161
11830     132
4637      126
12479      96
13285      91
9121       55

Top ASNs by total extreme prepend depth:
actual_prepend_asn
9318     6535
36992    4394
6306     1056
37069     990
4134      805
11830     792
12479     768
4637      756
13285     695
29465     528

Top ASNs by affected prefixes:
actual_prepend_asn
4134     160
9318      36
4637      14
3786      12
13285     11
36992     10
9121       5
6306       4
11830      4
37069      4

Top ASNs by affected peers:
actual_prepend_asn
29465    48
10361    44
6306     44
37069    44
24691    44
11830    43
36992    42
9318     37
12479    32
37287    23

Maximum prepend depth by actual ASN:
actual_prepend_asn
36992    13
24691    11
29465    11
10361    11
6713     10
237       9
13285     8
12479     8
3209      7
6128  

## 2.11.7 Long AS-Path Investigation

A long AS-path does not necessarily indicate a routing anomaly.

It may result from:
- legitimate multi-AS transit paths,
- AS-path prepending,
- or an unusual routing structure.

This investigation examines AS-path length and separates long paths
from paths dominated by AS-path prepending.

In [38]:
# ============================================================
# 2.11.7 LONG AS-PATH INVESTIGATION
# ============================================================

print("=" * 60)
print("LONG AS-PATH INVESTIGATION")
print("=" * 60)

LONG_PATH_THRESHOLD = 10

# ------------------------------------------------------------
# Identify long AS-path entries
# ------------------------------------------------------------

long_paths = df[
    df["as_path_length"] >= LONG_PATH_THRESHOLD
].copy()

print()
print(f"Total BGP entries        : {len(df):,}")
print(
    f"Long AS-path entries     : "
    f"{len(long_paths):,}"
)
print(
    f"Long AS-path percentage  : "
    f"{len(long_paths) / len(df) * 100:.2f}%"
)

# ------------------------------------------------------------
# AS-path length distribution
# ------------------------------------------------------------

print()
print("AS-path length distribution:")

path_length_distribution = (
    df["as_path_length"]
    .value_counts()
    .sort_index()
)

print(path_length_distribution.to_string())

# ------------------------------------------------------------
# Long-path statistics
# ------------------------------------------------------------

print()
print("Long AS-path statistics:")

print(
    long_paths["as_path_length"]
    .describe()
    .to_string()
)

# ------------------------------------------------------------
# Unique ASNs vs AS-path length
# ------------------------------------------------------------

long_paths["unique_asn_count"] = (
    long_paths["as_path_list"]
    .apply(lambda x: len(set(x)))
)

long_paths["repeat_excess"] = (
    long_paths["as_path_length"]
    - long_paths["unique_asn_count"]
)

print()
print("Long-path repeat statistics:")

print(
    long_paths["repeat_excess"]
    .describe()
    .to_string()
)

# ------------------------------------------------------------
# Long paths with significant repetition
# ------------------------------------------------------------

long_paths["has_significant_repetition"] = (
    long_paths["max_consecutive_repeat"] >= 5
)

significant_repeat_count = (
    long_paths["has_significant_repetition"]
    .sum()
)

print()
print(
    "Long paths with extreme consecutive "
    "prepending:"
)

print(
    f"Entries : {significant_repeat_count:,}"
)

print(
    f"Percentage of long paths : "
    f"{significant_repeat_count / len(long_paths) * 100:.2f}%"
)

# ------------------------------------------------------------
# Longest AS paths
# ------------------------------------------------------------

print()
print("Top 10 longest AS paths:")

longest_paths = (
    long_paths[
        [
            "prefix",
            "peer_ip",
            "peer_asn",
            "as_path",
            "as_path_length",
            "unique_asn_count",
            "max_consecutive_repeat"
        ]
    ]
    .sort_values(
        ["as_path_length", "max_consecutive_repeat"],
        ascending=False
    )
    .head(10)
)

print(
    longest_paths.to_string(
        index=False,
        max_colwidth=100
    )
)

# ------------------------------------------------------------
# Prefixes with longest observed paths
# ------------------------------------------------------------

print()
print("Top prefixes by maximum AS-path length:")

prefix_long_paths = (
    long_paths
    .groupby("prefix")
    .agg(
        observations=("prefix", "size"),
        max_path_length=("as_path_length", "max"),
        max_prepend_depth=("max_consecutive_repeat", "max"),
        unique_peers=("peer_ip", "nunique")
    )
    .sort_values(
        ["max_path_length", "observations"],
        ascending=False
    )
    .head(10)
)

print(prefix_long_paths.to_string())

# ------------------------------------------------------------
# Peers associated with long AS paths
# ------------------------------------------------------------

print()
print("Top peers associated with long AS paths:")

peer_long_paths = (
    long_paths
    .groupby("peer_ip")
    .agg(
        long_path_entries=("peer_ip", "size"),
        unique_prefixes=("prefix", "nunique"),
        max_path_length=("as_path_length", "max"),
        max_prepend_depth=("max_consecutive_repeat", "max")
    )
    .sort_values(
        ["long_path_entries", "max_path_length"],
        ascending=False
    )
    .head(10)
)

print(peer_long_paths.to_string())

# ------------------------------------------------------------
# Peer ASNs associated with long paths
# ------------------------------------------------------------

print()
print("Top peer ASNs associated with long AS paths:")

peer_asn_long_paths = (
    long_paths
    .groupby("peer_asn")
    .agg(
        long_path_entries=("peer_asn", "size"),
        unique_prefixes=("prefix", "nunique"),
        max_path_length=("as_path_length", "max")
    )
    .sort_values(
        ["long_path_entries", "max_path_length"],
        ascending=False
    )
    .head(10)
)

print(peer_asn_long_paths.to_string())

# ------------------------------------------------------------
# Long paths NOT dominated by prepending
# ------------------------------------------------------------

non_prepend_long_paths = long_paths[
    long_paths["max_consecutive_repeat"] < 5
].copy()

print()
print(
    "Long AS paths without extreme consecutive prepending:"
)

print(
    f"Entries : {len(non_prepend_long_paths):,}"
)

print(
    f"Percentage of long paths : "
    f"{len(non_prepend_long_paths) / len(long_paths) * 100:.2f}%"
)

# ------------------------------------------------------------
# Longest paths without extreme prepending
# ------------------------------------------------------------

print()
print(
    "Top 10 longest paths without extreme prepending:"
)

non_prepend_examples = (
    non_prepend_long_paths[
        [
            "prefix",
            "peer_ip",
            "peer_asn",
            "as_path",
            "as_path_length",
            "unique_asn_count",
            "max_consecutive_repeat"
        ]
    ]
    .sort_values(
        "as_path_length",
        ascending=False
    )
    .head(10)
)

print(
    non_prepend_examples.to_string(
        index=False,
        max_colwidth=100
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("LONG AS-PATH INVESTIGATION SUMMARY")
print("=" * 60)

print(
    f"Long-path threshold       : "
    f">= {LONG_PATH_THRESHOLD}"
)

print(
    f"Long AS-path entries      : "
    f"{len(long_paths):,}"
)

print(
    f"Long-path percentage      : "
    f"{len(long_paths) / len(df) * 100:.2f}%"
)

print(
    f"Maximum AS-path length    : "
    f"{long_paths['as_path_length'].max()}"
)

print(
    f"Long paths with extreme "
    f"prepending               : "
    f"{significant_repeat_count:,}"
)

print(
    f"Long paths without extreme "
    f"prepending               : "
    f"{len(non_prepend_long_paths):,}"
)

print()
print("Long AS-path investigation completed.")

LONG AS-PATH INVESTIGATION

Total BGP entries        : 100,000
Long AS-path entries     : 1,195
Long AS-path percentage  : 1.20%

AS-path length distribution:
as_path_length
1       102
2      5550
3     27792
4     31783
5     17438
6      9152
7      4708
8      1773
9       507
10      389
11      256
12       90
13       30
14       58
15       29
16       61
17      154
18      112
19       16

Long AS-path statistics:
count    1195.000000
mean       12.834310
std         3.055214
min        10.000000
25%        10.000000
50%        11.000000
75%        16.000000
max        19.000000

Long-path repeat statistics:
count    1195.000000
mean        7.726360
std         3.095022
min         1.000000
25%         5.000000
50%         7.000000
75%        12.000000
max        12.000000

Long paths with extreme consecutive prepending:
Entries : 1,135
Percentage of long paths : 94.98%

Top 10 longest AS paths:
        prefix        peer_ip  peer_asn                                          

## 2.11.8 Long AS-Path / Repetition Separation Analysis

In [39]:
# ============================================================
# 2.11.8 LONG AS-PATH / REPETITION SEPARATION ANALYSIS
# ============================================================

print("=" * 60)
print("LONG AS-PATH / REPETITION SEPARATION ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# Select long AS-path entries
# ------------------------------------------------------------

LONG_PATH_THRESHOLD = 10

long_paths = df[
    df["as_path_length"] >= LONG_PATH_THRESHOLD
].copy()

print()
print(f"Total BGP entries        : {len(df):,}")
print(f"Long AS-path entries     : {len(long_paths):,}")
print(
    f"Long AS-path percentage  : "
    f"{len(long_paths) / len(df) * 100:.2f}%"
)

# ------------------------------------------------------------
# Calculate unique ASN count
# ------------------------------------------------------------

long_paths["unique_asn_count"] = (
    long_paths["as_path_list"]
    .apply(lambda x: len(set(x)))
)

long_paths["repetition_count"] = (
    long_paths["as_path_length"]
    - long_paths["unique_asn_count"]
)

# ------------------------------------------------------------
# Classify long paths
#
# Category 1:
# Mostly unique ASNs
#
# Category 2:
# Moderate repetition
#
# Category 3:
# Heavy repetition / prepending
# ------------------------------------------------------------

def classify_long_path(row):

    path_length = row["as_path_length"]
    unique_asns = row["unique_asn_count"]
    max_repeat = row["max_consecutive_repeat"]

    repetition_ratio = (
        (path_length - unique_asns) / path_length
        if path_length > 0
        else 0
    )

    if max_repeat >= 5:
        return "Heavy repetition / prepending"

    elif repetition_ratio >= 0.30:
        return "Moderate repetition"

    else:
        return "Mostly unique ASNs"


long_paths["path_structure_class"] = (
    long_paths.apply(
        classify_long_path,
        axis=1
    )
)

# ------------------------------------------------------------
# Classification distribution
# ------------------------------------------------------------

print()
print("Long AS-path structural classification:")

classification_counts = (
    long_paths["path_structure_class"]
    .value_counts()
)

print(
    classification_counts.to_string()
)

print()

for category, count in classification_counts.items():

    percentage = (
        count / len(long_paths) * 100
    )

    print(
        f"{category:<35} : "
        f"{count:,} "
        f"({percentage:.2f}%)"
    )

# ------------------------------------------------------------
# Repetition statistics
# ------------------------------------------------------------

print()
print("Repetition statistics for long AS paths:")

print(
    long_paths[
        [
            "as_path_length",
            "unique_asn_count",
            "repetition_count",
            "max_consecutive_repeat"
        ]
    ]
    .describe()
    .to_string()
)

# ------------------------------------------------------------
# Compare path length against unique ASNs
# ------------------------------------------------------------

print()
print("Long paths with high repetition:")

high_repetition = long_paths[
    long_paths["repetition_count"] >= 5
].copy()

print(
    f"Entries with >= 5 repeated ASN positions : "
    f"{len(high_repetition):,}"
)

print(
    f"Percentage of long paths                 : "
    f"{len(high_repetition) / len(long_paths) * 100:.2f}%"
)

# ------------------------------------------------------------
# Long paths with many distinct ASNs
# ------------------------------------------------------------

print()
print("Long paths with high AS-path diversity:")

high_diversity = long_paths[
    long_paths["unique_asn_count"] >= 8
].copy()

print(
    f"Entries with >= 8 unique ASNs : "
    f"{len(high_diversity):,}"
)

print(
    f"Percentage of long paths     : "
    f"{len(high_diversity) / len(long_paths) * 100:.2f}%"
)

# ------------------------------------------------------------
# Examples of structurally different long paths
# ------------------------------------------------------------

print()
print("Examples of mostly unique-ASN long paths:")

mostly_unique = long_paths[
    long_paths["path_structure_class"]
    == "Mostly unique ASNs"
].sort_values(
    "unique_asn_count",
    ascending=False
)

print(
    mostly_unique[
        [
            "prefix",
            "peer_ip",
            "peer_asn",
            "as_path",
            "as_path_length",
            "unique_asn_count",
            "max_consecutive_repeat"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print()
print("Examples of heavily repeated long paths:")

heavily_repeated = long_paths[
    long_paths["path_structure_class"]
    == "Heavy repetition / prepending"
].sort_values(
    "max_consecutive_repeat",
    ascending=False
)

print(
    heavily_repeated[
        [
            "prefix",
            "peer_ip",
            "peer_asn",
            "as_path",
            "as_path_length",
            "unique_asn_count",
            "max_consecutive_repeat"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("LONG AS-PATH / REPETITION SEPARATION SUMMARY")
print("=" * 60)

print(
    f"Long AS-path entries          : "
    f"{len(long_paths):,}"
)

print(
    f"Mostly unique ASNs            : "
    f"{(long_paths['path_structure_class'] == 'Mostly unique ASNs').sum():,}"
)

print(
    f"Moderate repetition           : "
    f"{(long_paths['path_structure_class'] == 'Moderate repetition').sum():,}"
)

print(
    f"Heavy repetition / prepending : "
    f"{(long_paths['path_structure_class'] == 'Heavy repetition / prepending').sum():,}"
)

print()
print(
    "Long AS-path / repetition separation analysis completed."
)

LONG AS-PATH / REPETITION SEPARATION ANALYSIS

Total BGP entries        : 100,000
Long AS-path entries     : 1,195
Long AS-path percentage  : 1.20%

Long AS-path structural classification:
path_structure_class
Heavy repetition / prepending    1135
Moderate repetition                56
Mostly unique ASNs                  4

Heavy repetition / prepending       : 1,135 (94.98%)
Moderate repetition                 : 56 (4.69%)
Mostly unique ASNs                  : 4 (0.33%)

Repetition statistics for long AS paths:
       as_path_length  unique_asn_count  repetition_count  max_consecutive_repeat
count     1195.000000       1195.000000       1195.000000             1195.000000
mean        12.834310          5.107950          7.726360                8.601674
std          3.055214          1.017055          3.095022                3.207821
min         10.000000          2.000000          1.000000                2.000000
25%         10.000000          4.000000          5.000000                

## 2.11.9 AS-Path Structural Anomaly Summary

This section consolidates the major AS-path structural findings identified
through Sections 2.11.1–2.11.8.

| Metric | Result | Interpretation |
|---|---:|---|
| Total BGP entries | 100,000 | Dataset size |
| Entries with repeated ASNs | 17,419 (17.42%) | ASN repetition is common |
| Entries with AS-path prepending | 17,417 (17.42%) | Prepending is a major contributor |
| Non-consecutive ASN repetition | 2 | Rare structural pattern |
| Origin ASN mismatches | 41 (0.04%) | Very low inconsistency rate |
| Long AS-paths (≥10) | 1,195 (1.20%) | Relatively uncommon |
| Long paths with heavy repetition | 1,135 (94.98%) | Most long paths are caused by repetition |
| Long paths with moderate repetition | 56 (4.69%) | Small secondary group |
| Long paths with mostly unique ASNs | 4 (0.33%) | Genuine path diversity is rare |
| Extreme prepending (≥5) | 2,943 (2.94%) | Significant subset of all entries |
| Maximum prepend depth | 13 | Strong prepending observed |
| Maximum AS-path length | 19 | Longest observed path |

### Key findings

- **AS-path repetition is common:** 17.42% of BGP entries contain repeated ASNs.
- **Extreme prepending is significant:** 2,943 entries contain at least five consecutive repetitions.
- **Long AS-paths are overwhelmingly repetition-driven:** 94.98% of the 1,195 long paths contain heavy repetition/prepending.
- **Genuine AS-path diversity is rare:** only 4 long paths were classified as mostly unique-ASN paths.
- **Origin ASN inconsistency is uncommon:** only 41 entries (0.04%) were identified as mismatches.
- **The maximum observed prepend depth is 13**, while the maximum AS-path length is 19.

### Overall assessment

The observed AS-path anomalies are dominated by **ASN repetition and prepending**,
rather than unusually high transit-AS diversity.

This indicates that AS-path length alone should not be interpreted as a measure
of routing-path complexity. For BGP analytics, **unique ASN count, repetition
depth, and prepend structure should be evaluated together with AS-path length.**

###

## 2.12 BGP Path Attribute Profiling

BGP route selection is influenced by multiple path attributes beyond the AS-path.

This section profiles the consistency, distribution, and potential anomalies
of the major BGP path attributes observed in the dataset.

We will examine:

- Next-hop consistency
- Origin attribute
- MED
- Local Preference, where available
- Communities, where available
- Attribute combinations and inconsistencies
- Unusual or conflicting attribute patterns

The objective is to understand the quality and structural characteristics
of BGP path attributes before applying route-selection or anomaly-detection
logic.

## 2.12.1 BGP Attribute Availability / Inventory

Before profiling individual BGP path attributes, we first identify which
attributes are actually present in the parsed dataset.

This prevents assumptions about field names or attributes that may not be
available in the MRT/BGP collector data.

We will identify:

- Attribute names
- Data types
- Number of populated values
- Number of missing values
- Number of unique values

In [41]:
# ============================================================
# 2.12.1 BGP ATTRIBUTE AVAILABILITY / INVENTORY
# ============================================================

print("=" * 60)
print("BGP ATTRIBUTE AVAILABILITY / INVENTORY")
print("=" * 60)

total_entries = len(df)

print()
print(f"Total BGP entries : {total_entries:,}")
print()

# ------------------------------------------------------------
# Helper: safely count unique values
# ------------------------------------------------------------

def safe_unique_count(series):
    """
    Count unique values safely.

    Some BGP attributes are list-valued and therefore cannot
    be processed directly by pandas.nunique().
    """

    non_null_values = series.dropna()

    if len(non_null_values) == 0:
        return 0

    try:
        return non_null_values.nunique()

    except TypeError:
        # Convert list / dict / other unhashable objects
        # into a comparable string representation.
        return (
            non_null_values
            .apply(lambda x: repr(x))
            .nunique()
        )


# ------------------------------------------------------------
# Attribute inventory
# ------------------------------------------------------------

attribute_inventory = []

for column in df.columns:

    non_null = df[column].notna().sum()
    null_count = df[column].isna().sum()
    unique_count = safe_unique_count(df[column])

    attribute_inventory.append({
        "attribute": column,
        "dtype": str(df[column].dtype),
        "populated": non_null,
        "missing": null_count,
        "population_pct": (
            non_null / total_entries * 100
            if total_entries
            else 0
        ),
        "unique_values": unique_count
    })


attribute_inventory = pd.DataFrame(
    attribute_inventory
)

# ------------------------------------------------------------
# Display inventory
# ------------------------------------------------------------

print("Dataset attribute inventory:")
print()

print(
    attribute_inventory
    .sort_values("attribute")
    .to_string(index=False)
)

# ------------------------------------------------------------
# Identify likely BGP path attributes
# ------------------------------------------------------------

print()
print("=" * 60)
print("LIKELY BGP PATH ATTRIBUTES")
print("=" * 60)

candidate_keywords = [
    "next",
    "hop",
    "origin",
    "med",
    "metric",
    "local",
    "pref",
    "community",
    "as_path",
    "peer"
]

likely_attributes = []

for column in df.columns:

    column_lower = column.lower()

    if any(
        keyword in column_lower
        for keyword in candidate_keywords
    ):
        likely_attributes.append(column)

print()

for column in likely_attributes:
    print(f"- {column}")

# ------------------------------------------------------------
# Attribute availability summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("ATTRIBUTE AVAILABILITY SUMMARY")
print("=" * 60)

print(
    f"Total dataframe columns : "
    f"{len(df.columns):,}"
)

print(
    f"Likely BGP attributes   : "
    f"{len(likely_attributes):,}"
)

print()
print(
    "Attribute inventory completed."
)

BGP ATTRIBUTE AVAILABILITY / INVENTORY

Total BGP entries : 100,000

Dataset attribute inventory:

                 attribute  dtype  populated  missing  population_pct  unique_values
                   as_path    str     100000        0           100.0          14620
       as_path_has_prepend   bool     100000        0           100.0              2
        as_path_has_repeat   bool     100000        0           100.0              2
          as_path_last_asn    str     100000        0           100.0            301
         as_path_last_asns object     100000        0           100.0            301
            as_path_length  int64     100000        0           100.0             19
              as_path_list object     100000        0           100.0          14620
    has_consecutive_repeat   bool     100000        0           100.0              2
has_non_consecutive_repeat   bool     100000        0           100.0              2
                local_pref  int64     100000       

## 2.12.2 Next-Hop Consistency Profiling

The BGP next-hop attribute determines the next-hop address associated with
a route candidate.

This section profiles next-hop consistency across the dataset and,
in particular, examines how next-hop values vary for the same prefix.

We will examine:

- Number of unique next-hops in the dataset
- Number of unique next-hops per prefix
- Prefixes with multiple next-hops
- Prefixes with unusually high next-hop diversity
- Relationship between next-hop diversity and peer diversity
- Next-hop consistency across observations

The objective is to determine whether next-hop variation is a normal
characteristic of the observed BGP data or whether unusual patterns
require further investigation.

In [42]:
# ============================================================
# 2.12.2 NEXT-HOP CONSISTENCY PROFILING
# ============================================================

print("=" * 60)
print("NEXT-HOP CONSISTENCY PROFILING")
print("=" * 60)

total_entries = len(df)

print()
print(f"Total BGP entries : {total_entries:,}")

# ------------------------------------------------------------
# 1. Dataset-wide next-hop inventory
# ------------------------------------------------------------

unique_next_hops = df["next_hop"].nunique(dropna=True)
missing_next_hops = df["next_hop"].isna().sum()

print()
print("Dataset next-hop inventory:")
print(f"Unique next-hops : {unique_next_hops:,}")
print(f"Missing next-hops: {missing_next_hops:,}")

# ------------------------------------------------------------
# 2. Unique next-hops per prefix
# ------------------------------------------------------------

next_hop_profile = (
    df.groupby("prefix")
      .agg(
          route_candidates=("prefix", "size"),
          unique_next_hops=("next_hop", "nunique"),
          unique_peers=("peer_ip", "nunique")
      )
      .sort_values(
          "unique_next_hops",
          ascending=False
      )
)

print()
print("Next-hop diversity statistics per prefix:")
print(
    next_hop_profile[
        ["route_candidates", "unique_next_hops", "unique_peers"]
    ].describe()
)

# ------------------------------------------------------------
# 3. Prefixes with multiple next-hops
# ------------------------------------------------------------

multi_next_hop = next_hop_profile[
    next_hop_profile["unique_next_hops"] > 1
].copy()

single_next_hop = next_hop_profile[
    next_hop_profile["unique_next_hops"] == 1
].copy()

print()
print("Next-hop consistency classification:")
print(
    f"Prefixes with one next-hop  : "
    f"{len(single_next_hop):,}"
)

print(
    f"Prefixes with multiple next-hops : "
    f"{len(multi_next_hop):,}"
)

print(
    f"Percentage with multiple next-hops : "
    f"{len(multi_next_hop) / len(next_hop_profile) * 100:.2f}%"
)

# ------------------------------------------------------------
# 4. Top prefixes by next-hop diversity
# ------------------------------------------------------------

print()
print("Top 10 prefixes by unique next-hop count:")
print()

top_next_hop_prefixes = (
    next_hop_profile
    .sort_values(
        ["unique_next_hops", "route_candidates"],
        ascending=False
    )
    .head(10)
)

print(top_next_hop_prefixes.to_string())

# ------------------------------------------------------------
# 5. Check whether next-hop diversity follows peer diversity
# ------------------------------------------------------------

print()
print("Next-hop vs peer diversity:")

next_hop_peer_relationship = (
    next_hop_profile[
        ["unique_next_hops", "unique_peers"]
    ]
)

print(
    next_hop_peer_relationship.describe()
)

# ------------------------------------------------------------
# 6. Prefixes where next-hop count differs from peer count
# ------------------------------------------------------------

next_hop_peer_mismatch = next_hop_profile[
    next_hop_profile["unique_next_hops"]
    !=
    next_hop_profile["unique_peers"]
].copy()

print()
print(
    "Prefixes where unique next-hops != unique peers : "
    f"{len(next_hop_peer_mismatch):,}"
)

if len(next_hop_peer_mismatch) > 0:
    print()
    print("Top examples:")
    print(
        next_hop_peer_mismatch
        .sort_values(
            "route_candidates",
            ascending=False
        )
        .head(10)
        .to_string()
    )

# ------------------------------------------------------------
# 7. Most frequently observed next-hops
# ------------------------------------------------------------

print()
print("Top 10 next-hops by observation count:")
print()

print(
    df["next_hop"]
    .value_counts(dropna=False)
    .head(10)
    .to_string()
)

# ------------------------------------------------------------
# 8. Prefix / next-hop examples
# ------------------------------------------------------------

print()
print("Examples of prefixes with multiple next-hops:")
print()

if len(multi_next_hop) > 0:

    example_prefixes = (
        multi_next_hop
        .head(10)
        .index
    )

    example_view = (
        df[df["prefix"].isin(example_prefixes)]
        [
            [
                "prefix",
                "peer_ip",
                "peer_asn",
                "next_hop"
            ]
        ]
        .sort_values(
            ["prefix", "next_hop", "peer_ip"]
        )
    )

    print(
        example_view.to_string(index=False)
    )

else:
    print("No prefixes with multiple next-hops were observed.")

# ------------------------------------------------------------
# 9. Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("NEXT-HOP CONSISTENCY PROFILING SUMMARY")
print("=" * 60)

print(
    f"Total BGP entries              : "
    f"{total_entries:,}"
)

print(
    f"Unique next-hops               : "
    f"{unique_next_hops:,}"
)

print(
    f"Missing next-hops              : "
    f"{missing_next_hops:,}"
)

print(
    f"Prefixes with one next-hop     : "
    f"{len(single_next_hop):,}"
)

print(
    f"Prefixes with multiple next-hops : "
    f"{len(multi_next_hop):,}"
)

print(
    f"Next-hop diversity rate        : "
    f"{len(multi_next_hop) / len(next_hop_profile) * 100:.2f}%"
)

print()
print("No records were modified or removed.")
print("Next-hop consistency profiling completed.")

NEXT-HOP CONSISTENCY PROFILING

Total BGP entries : 100,000

Dataset next-hop inventory:
Unique next-hops : 59
Missing next-hops: 0

Next-hop diversity statistics per prefix:
       route_candidates  unique_next_hops  unique_peers
count       2316.000000       2316.000000   2316.000000
mean          43.177893         43.177893     43.177893
std            4.078745          4.078745      4.078745
min            1.000000          1.000000      1.000000
25%           43.000000         43.000000     43.000000
50%           43.000000         43.000000     43.000000
75%           44.000000         44.000000     44.000000
max           51.000000         51.000000     51.000000

Next-hop consistency classification:
Prefixes with one next-hop  : 7
Prefixes with multiple next-hops : 2,309
Percentage with multiple next-hops : 99.70%

Top 10 prefixes by unique next-hop count:

                route_candidates  unique_next_hops  unique_peers
prefix                                                   

## 2.12.3 Origin Attribute Profiling

The BGP ORIGIN attribute indicates how the route was originally introduced
into BGP.

This section profiles the ORIGIN attribute distribution and examines its
relationship with prefixes, peers, and AS-path structure.

We will examine:

- ORIGIN attribute values and distribution
- ORIGIN distribution by prefix
- ORIGIN distribution by peer
- ORIGIN distribution by peer ASN
- Relationship between ORIGIN and AS-path length
- Unusual ORIGIN patterns

The objective is to determine whether the ORIGIN attribute shows a
consistent and expected distribution across the dataset.

In [43]:
# ============================================================
# 2.12.3 ORIGIN ATTRIBUTE PROFILING
# ============================================================

print("=" * 60)
print("ORIGIN ATTRIBUTE PROFILING")
print("=" * 60)

total_entries = len(df)

# ------------------------------------------------------------
# 1. ORIGIN attribute inventory
# ------------------------------------------------------------

origin_counts = (
    df["origin"]
    .value_counts(dropna=False)
)

print()
print("ORIGIN attribute distribution:")
print()

print(origin_counts.to_string())

print()

for origin_value, count in origin_counts.items():

    percentage = (
        count / total_entries * 100
    )

    print(
        f"{str(origin_value):<10} : "
        f"{count:,} "
        f"({percentage:.2f}%)"
    )

# ------------------------------------------------------------
# 2. Unique ORIGIN values per prefix
# ------------------------------------------------------------

origin_prefix_profile = (
    df.groupby("prefix")
      .agg(
          route_candidates=("prefix", "size"),
          unique_origins=("origin", "nunique"),
          unique_peers=("peer_ip", "nunique")
      )
)

print()
print("ORIGIN diversity per prefix:")
print()

print(
    origin_prefix_profile[
        [
            "route_candidates",
            "unique_origins",
            "unique_peers"
        ]
    ]
    .describe()
    .to_string()
)

# ------------------------------------------------------------
# 3. Prefixes with multiple ORIGIN values
# ------------------------------------------------------------

multi_origin_prefixes = (
    origin_prefix_profile[
        origin_prefix_profile["unique_origins"] > 1
    ]
)

print()
print(
    f"Prefixes with multiple ORIGIN values : "
    f"{len(multi_origin_prefixes):,}"
)

print(
    f"Percentage of prefixes              : "
    f"{len(multi_origin_prefixes) / len(origin_prefix_profile) * 100:.2f}%"
)

# ------------------------------------------------------------
# 4. ORIGIN distribution by peer ASN
# ------------------------------------------------------------

print()
print("ORIGIN distribution by peer ASN:")

origin_peer_asn = (
    pd.crosstab(
        df["peer_asn"],
        df["origin"]
    )
)

print(
    origin_peer_asn.to_string()
)

# ------------------------------------------------------------
# 5. ORIGIN vs AS-path length
# ------------------------------------------------------------

print()
print("AS-path length by ORIGIN:")

origin_path_stats = (
    df.groupby("origin")["as_path_length"]
      .agg(
          entries="count",
          average="mean",
          minimum="min",
          maximum="max"
      )
)

print(
    origin_path_stats.to_string()
)

# ------------------------------------------------------------
# 6. ORIGIN vs AS-path repetition
# ------------------------------------------------------------

print()
print("AS-path repetition by ORIGIN:")

origin_repeat_stats = (
    df.groupby("origin")["max_consecutive_repeat"]
      .agg(
          average="mean",
          maximum="max"
      )
)

print(
    origin_repeat_stats.to_string()
)

# ------------------------------------------------------------
# 7. Most common prefix / ORIGIN combinations
# ------------------------------------------------------------

print()
print("Most common prefix / ORIGIN combinations:")

prefix_origin_counts = (
    df.groupby(
        ["prefix", "origin"]
    )
    .size()
    .reset_index(
        name="entries"
    )
    .sort_values(
        "entries",
        ascending=False
    )
    .head(10)
)

print(
    prefix_origin_counts.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 8. Most common peer / ORIGIN combinations
# ------------------------------------------------------------

print()
print("Most common peer / ORIGIN combinations:")

peer_origin_counts = (
    df.groupby(
        ["peer_ip", "origin"]
    )
    .size()
    .reset_index(
        name="entries"
    )
    .sort_values(
        "entries",
        ascending=False
    )
    .head(10)
)

print(
    peer_origin_counts.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 9. Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("ORIGIN ATTRIBUTE PROFILING SUMMARY")
print("=" * 60)

print(
    f"Total BGP entries             : "
    f"{total_entries:,}"
)

print(
    f"Unique ORIGIN values          : "
    f"{df['origin'].nunique(dropna=True):,}"
)

print(
    f"Prefixes with multiple ORIGINs : "
    f"{len(multi_origin_prefixes):,}"
)

print(
    f"Maximum AS-path length        : "
    f"{df['as_path_length'].max()}"
)

print()
print("No records were modified or removed.")
print("ORIGIN attribute profiling completed.")

ORIGIN ATTRIBUTE PROFILING

ORIGIN attribute distribution:

origin
IGP           96582
INCOMPLETE     3418

IGP        : 96,582 (96.58%)
INCOMPLETE : 3,418 (3.42%)

ORIGIN diversity per prefix:

       route_candidates  unique_origins  unique_peers
count       2316.000000     2316.000000   2316.000000
mean          43.177893        1.987478     43.177893
std            4.078745        0.111221      4.078745
min            1.000000        1.000000      1.000000
25%           43.000000        2.000000     43.000000
50%           43.000000        2.000000     43.000000
75%           44.000000        2.000000     44.000000
max           51.000000        2.000000     51.000000

Prefixes with multiple ORIGIN values : 2,287
Percentage of prefixes              : 98.75%

ORIGIN distribution by peer ASN:
origin     IGP  INCOMPLETE
peer_asn                  
852       2296           0
1403      2250          36
3333      2218          84
3549      2282           0
4608      2269          24
7018 

2.12.3.1 ORIGIN Pattern / Peer Attribution Analysis

In [44]:
# ============================================================
# 2.12.3.1 ORIGIN Pattern / Peer Attribution Analysis
# ============================================================

print("=" * 60)
print("ORIGIN PATTERN / PEER ATTRIBUTION ANALYSIS")
print("=" * 60)

total_entries = len(df)

# ------------------------------------------------------------
# 1. ORIGIN distribution by peer
# ------------------------------------------------------------

print()
print("ORIGIN distribution by peer:")
print()

peer_origin = pd.crosstab(
    df["peer_ip"],
    df["origin"]
)

print(peer_origin.to_string())

# ------------------------------------------------------------
# 2. ORIGIN distribution by peer ASN
# ------------------------------------------------------------

print()
print("ORIGIN distribution by peer ASN:")
print()

peer_asn_origin = pd.crosstab(
    df["peer_asn"],
    df["origin"]
)

print(peer_asn_origin.to_string())

# ------------------------------------------------------------
# 3. Calculate INCOMPLETE percentage by peer ASN
# ------------------------------------------------------------

peer_asn_origin_analysis = (
    df.groupby("peer_asn")
    .agg(
        total_entries=("origin", "size"),
        incomplete_entries=(
            "origin",
            lambda x: (x == "INCOMPLETE").sum()
        ),
        igp_entries=(
            "origin",
            lambda x: (x == "IGP").sum()
        )
    )
)

peer_asn_origin_analysis["incomplete_pct"] = (
    peer_asn_origin_analysis["incomplete_entries"]
    / peer_asn_origin_analysis["total_entries"]
    * 100
)

peer_asn_origin_analysis["igp_pct"] = (
    peer_asn_origin_analysis["igp_entries"]
    / peer_asn_origin_analysis["total_entries"]
    * 100
)

# ------------------------------------------------------------
# 4. Top peer ASNs by INCOMPLETE count
# ------------------------------------------------------------

print()
print("Top 10 peer ASNs by INCOMPLETE count:")
print()

print(
    peer_asn_origin_analysis
    .sort_values(
        "incomplete_entries",
        ascending=False
    )
    .head(10)
    .to_string()
)

# ------------------------------------------------------------
# 5. Peer ASNs with highest INCOMPLETE percentage
# ------------------------------------------------------------

print()
print("Top 10 peer ASNs by INCOMPLETE percentage:")
print()

print(
    peer_asn_origin_analysis[
        peer_asn_origin_analysis["total_entries"] >= 10
    ]
    .sort_values(
        "incomplete_pct",
        ascending=False
    )
    .head(10)
    .to_string()
)

# ------------------------------------------------------------
# 6. Identify ORIGIN patterns
# ------------------------------------------------------------

def classify_origin_pattern(row):
    if row["incomplete_entries"] == 0:
        return "IGP only"
    
    if row["igp_entries"] == 0:
        return "INCOMPLETE only"
    
    return "Mixed"


peer_asn_origin_analysis["origin_pattern"] = (
    peer_asn_origin_analysis.apply(
        classify_origin_pattern,
        axis=1
    )
)

print()
print("Peer ASN ORIGIN pattern classification:")
print()

print(
    peer_asn_origin_analysis[
        ["total_entries",
         "igp_entries",
         "incomplete_entries",
         "igp_pct",
         "incomplete_pct",
         "origin_pattern"]
    ]
    .sort_values(
        "incomplete_entries",
        ascending=False
    )
    .to_string()
)

# ------------------------------------------------------------
# 7. Pattern summary
# ------------------------------------------------------------

print()
print("ORIGIN pattern summary:")
print()

pattern_summary = (
    peer_asn_origin_analysis["origin_pattern"]
    .value_counts()
)

print(pattern_summary.to_string())

# ------------------------------------------------------------
# 8. Examples of INCOMPLETE routes
# ------------------------------------------------------------

print()
print("Examples of INCOMPLETE ORIGIN routes:")
print()

incomplete_examples = df[
    df["origin"] == "INCOMPLETE"
][
    [
        "prefix",
        "peer_ip",
        "peer_asn",
        "origin",
        "as_path",
        "as_path_length"
    ]
].head(20)

print(
    incomplete_examples.to_string(index=False)
)

# ------------------------------------------------------------
# 9. Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("ORIGIN PATTERN / PEER ATTRIBUTION SUMMARY")
print("=" * 60)

print(
    f"Total BGP entries              : "
    f"{total_entries:,}"
)

print(
    f"Unique peer ASNs               : "
    f"{df['peer_asn'].nunique():,}"
)

print(
    f"Peer ASNs with IGP only        : "
    f"{(peer_asn_origin_analysis['origin_pattern'] == 'IGP only').sum():,}"
)

print(
    f"Peer ASNs with INCOMPLETE only : "
    f"{(peer_asn_origin_analysis['origin_pattern'] == 'INCOMPLETE only').sum():,}"
)

print(
    f"Peer ASNs with mixed ORIGIN    : "
    f"{(peer_asn_origin_analysis['origin_pattern'] == 'Mixed').sum():,}"
)

print()
print("No records were modified or removed.")
print(
    "ORIGIN pattern / peer attribution analysis completed."
)

ORIGIN PATTERN / PEER ATTRIBUTION ANALYSIS

ORIGIN distribution by peer:

origin            IGP  INCOMPLETE
peer_ip                          
102.208.105.2    2195          86
102.217.156.3     155           1
103.147.22.50      79           2
103.212.68.10    2276          21
109.248.43.5      173           1
118.67.200.8       71           0
12.0.1.63        2235          46
154.11.12.212    2296           0
161.129.152.2    2282           0
165.16.221.66    2272          33
176.12.110.8     2269          26
178.208.11.4      681           4
185.102.84.232   2295           0
185.142.156.156     1           2
185.152.34.255    218           0
185.193.84.191   2279          20
192.248.121.251   111          15
193.0.0.56       2218          84
193.148.251.1    2284           3
193.150.22.240   2273          19
193.160.39.1     2239          57
193.163.86.231   2298           2
193.33.94.231       1           0
193.33.94.241       1           0
193.33.94.251    2220          77
194.28.9

## 2.12.3.2 ORIGIN / AS-Path Relationship Analysis

This section examines the relationship between the BGP ORIGIN attribute
and AS-path structure.

The analysis compares:

- ORIGIN distribution by AS-path length
- Average and maximum AS-path length by ORIGIN
- AS-path repetition by ORIGIN
- Extreme prepending by ORIGIN
- Long AS-paths by ORIGIN
- AS-path structural characteristics of IGP and INCOMPLETE routes

The objective is to identify observable relationships between ORIGIN and
AS-path structure.

These results describe association and structural relationships only.
They do not establish causality between the attributes.

In [50]:
# ============================================================
# 2.12.3.2 ORIGIN / AS-PATH RELATIONSHIP ANALYSIS
# ============================================================

print("=" * 60)
print("ORIGIN / AS-PATH RELATIONSHIP ANALYSIS")
print("=" * 60)

total_entries = len(df)

# ------------------------------------------------------------
# 1. ORIGIN vs AS-path length distribution
# ------------------------------------------------------------

print()
print("AS-path length distribution by ORIGIN:")
print()

origin_path_length = pd.crosstab(
    df["as_path_length"],
    df["origin"]
)

print(
    origin_path_length.to_string()
)

# ------------------------------------------------------------
# 2. AS-path statistics by ORIGIN
# ------------------------------------------------------------

print()
print("AS-path statistics by ORIGIN:")
print()

origin_path_stats = (
    df.groupby("origin")["as_path_length"]
    .agg(
        entries="count",
        average="mean",
        median="median",
        minimum="min",
        maximum="max"
    )
)

print(
    origin_path_stats.to_string()
)

# ------------------------------------------------------------
# 3. Unique ASN count by ORIGIN
# ------------------------------------------------------------

# This field was created earlier in Section 2.11.
# Calculate it here as well so this subsection is self-contained.

origin_analysis = df.copy()

origin_analysis["unique_asn_count"] = (
    origin_analysis["as_path_list"]
    .apply(lambda x: len(set(x)))
)

origin_analysis["repetition_count"] = (
    origin_analysis["as_path_length"]
    - origin_analysis["unique_asn_count"]
)

print()
print("Unique ASN statistics by ORIGIN:")
print()

unique_asn_stats = (
    origin_analysis
    .groupby("origin")["unique_asn_count"]
    .agg(
        average="mean",
        median="median",
        minimum="min",
        maximum="max"
    )
)

print(
    unique_asn_stats.to_string()
)

# ------------------------------------------------------------
# 4. Repetition statistics by ORIGIN
# ------------------------------------------------------------

print()
print("AS-path repetition statistics by ORIGIN:")
print()

repetition_stats = (
    origin_analysis
    .groupby("origin")["repetition_count"]
    .agg(
        average="mean",
        median="median",
        minimum="min",
        maximum="max"
    )
)

print(
    repetition_stats.to_string()
)

# ------------------------------------------------------------
# 5. Extreme prepending by ORIGIN
# ------------------------------------------------------------

EXTREME_PREPEND_THRESHOLD = 5

origin_analysis["extreme_prepending"] = (
    origin_analysis["max_consecutive_repeat"]
    >= EXTREME_PREPEND_THRESHOLD
)

extreme_by_origin = (
    origin_analysis
    .groupby("origin")["extreme_prepending"]
    .agg(
        entries="size",
        extreme_entries="sum"
    )
)

extreme_by_origin["extreme_pct"] = (
    extreme_by_origin["extreme_entries"]
    / extreme_by_origin["entries"]
    * 100
)

print()
print(
    "Extreme prepending by ORIGIN:"
)

print(
    extreme_by_origin.to_string()
)

# ------------------------------------------------------------
# 6. Long AS-paths by ORIGIN
# ------------------------------------------------------------

LONG_PATH_THRESHOLD = 10

origin_analysis["long_path"] = (
    origin_analysis["as_path_length"]
    >= LONG_PATH_THRESHOLD
)

long_by_origin = (
    origin_analysis
    .groupby("origin")["long_path"]
    .agg(
        entries="size",
        long_path_entries="sum"
    )
)

long_by_origin["long_path_pct"] = (
    long_by_origin["long_path_entries"]
    / long_by_origin["entries"]
    * 100
)

print()
print(
    f"Long AS-paths (>= {LONG_PATH_THRESHOLD}) by ORIGIN:"
)

print(
    long_by_origin.to_string()
)

# ------------------------------------------------------------
# 7. Long paths and extreme prepending by ORIGIN
# ------------------------------------------------------------

long_extreme = origin_analysis[
    origin_analysis["long_path"]
    &
    origin_analysis["extreme_prepending"]
]

long_extreme_by_origin = (
    long_extreme
    .groupby("origin")
    .size()
    .rename("long_and_extreme_entries")
)

print()
print(
    "Long AS-paths with extreme prepending by ORIGIN:"
)

print(
    long_extreme_by_origin.to_string()
)

# ------------------------------------------------------------
# ORIGIN / AS-path structural summary
# ------------------------------------------------------------

# ------------------------------------------------------------
# ORIGIN / AS-PATH STRUCTURAL SUMMARY
# ------------------------------------------------------------
# ------------------------------------------------------------
# ORIGIN / AS-PATH STRUCTURAL SUMMARY
# ------------------------------------------------------------

print()
print("=" * 60)
print("ORIGIN / AS-PATH STRUCTURAL SUMMARY")
print("=" * 60)

# ------------------------------------------------------------
# Prepare masks
# ------------------------------------------------------------

igp_mask = df["origin"] == "IGP"
incomplete_mask = df["origin"] == "INCOMPLETE"

igp_entries = igp_mask.sum()
incomplete_entries = incomplete_mask.sum()

igp_avg_path = df.loc[
    igp_mask,
    "as_path_length"
].mean()

incomplete_avg_path = df.loc[
    incomplete_mask,
    "as_path_length"
].mean()

igp_max_path = df.loc[
    igp_mask,
    "as_path_length"
].max()

incomplete_max_path = df.loc[
    incomplete_mask,
    "as_path_length"
].max()

igp_extreme = (
    igp_mask
    & (df["max_consecutive_repeat"] >= 5)
).sum()

incomplete_extreme = (
    incomplete_mask
    & (df["max_consecutive_repeat"] >= 5)
).sum()

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print()
print("Key structural observations:")
print()

print("IGP:")
print(f"  Entries                : {igp_entries:,}")
print(f"  Average AS-path length : {igp_avg_path:.2f}")
print(f"  Maximum AS-path length : {igp_max_path}")
print(f"  Extreme prepending     : {igp_extreme:,}")

print()

print("INCOMPLETE:")
print(f"  Entries                : {incomplete_entries:,}")
print(f"  Average AS-path length : {incomplete_avg_path:.2f}")
print(f"  Maximum AS-path length : {incomplete_max_path}")
print(f"  Extreme prepending     : {incomplete_extreme:,}")

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("ORIGIN / AS-PATH RELATIONSHIP SUMMARY")
print("=" * 60)

print(f"Total BGP entries         : {len(df):,}")
print(
    f"ORIGIN values analyzed    : "
    f"{df['origin'].nunique()}"
)
print("Long-path threshold       : >= 10")
print("Extreme-prepend threshold : >= 5")

print()
print("No causal relationship is inferred from these results.")

print(
    "ORIGIN / AS-path relationship analysis completed."
)

ORIGIN / AS-PATH RELATIONSHIP ANALYSIS

AS-path length distribution by ORIGIN:

origin            IGP  INCOMPLETE
as_path_length                   
1                 102           0
2                5256         294
3               26310        1482
4               31010         773
5               17081         357
6                8881         271
7                4613          95
8                1728          45
9                 472          35
10                355          34
11                244          12
12                 89           1
13                 24           6
14                 54           4
15                 28           1
16                 61           0
17                146           8
18                112           0
19                 16           0

AS-path statistics by ORIGIN:

            entries   average  median  minimum  maximum
origin                                                 
IGP           96582  4.319708     4.0        1       19
INCOMP

## 2.12.4 MED Profiling

The BGP MULTI_EXIT_DISC (MED) attribute provides a mechanism for
influencing route selection between multiple entry points into an
autonomous system.

This section profiles the MED attribute distribution and examines its
relationship with prefixes, peers, peer ASNs, and AS-path structure.

We will examine:

- MED availability and value distribution
- MED frequency and diversity
- MED distribution by prefix
- MED distribution by peer and peer ASN
- Common and unusual MED values
- MED relationship with AS-path length
- MED relationship with ORIGIN
- Potentially unusual MED patterns

The objective is to establish the structural characteristics of MED
in the dataset before applying route-selection or anomaly-detection
logic.

The analysis identifies observable relationships only and does not
infer causality or actual BGP route-selection decisions.

In [51]:
# ============================================================
# 2.12.4 MED PROFILING
# ============================================================

print("=" * 60)
print("MED PROFILING")
print("=" * 60)

total_entries = len(df)

# ------------------------------------------------------------
# 1. MED availability
# ------------------------------------------------------------

med_non_null = df["med"].notna().sum()
med_null = df["med"].isna().sum()

print()
print("MED availability:")
print()

print(
    f"Total BGP entries : {total_entries:,}"
)

print(
    f"MED populated     : {med_non_null:,} "
    f"({med_non_null / total_entries * 100:.2f}%)"
)

print(
    f"MED missing       : {med_null:,} "
    f"({med_null / total_entries * 100:.2f}%)"
)

# ------------------------------------------------------------
# 2. MED value statistics
# ------------------------------------------------------------

if med_non_null > 0:

    print()
    print("MED statistics:")
    print()

    med_stats = df["med"].describe()

    print(
        med_stats.to_string()
    )

# ------------------------------------------------------------
# 3. MED value distribution
# ------------------------------------------------------------

if med_non_null > 0:

    print()
    print("Most common MED values:")
    print()

    med_distribution = (
        df["med"]
        .value_counts(dropna=True)
        .head(20)
    )

    print(
        med_distribution.to_string()
    )

# ------------------------------------------------------------
# 4. MED diversity
# ------------------------------------------------------------

unique_med_values = (
    df["med"]
    .dropna()
    .nunique()
)

print()
print(
    f"Unique MED values : {unique_med_values:,}"
)

# ------------------------------------------------------------
# 5. MED distribution by peer ASN
# ------------------------------------------------------------

if med_non_null > 0:

    print()
    print("MED statistics by peer ASN:")
    print()

    med_by_peer_asn = (
        df.dropna(subset=["med"])
        .groupby("peer_asn")["med"]
        .agg(
            entries="count",
            unique_med_values="nunique",
            average="mean",
            minimum="min",
            maximum="max"
        )
        .sort_values(
            "entries",
            ascending=False
        )
    )

    print(
        med_by_peer_asn.to_string()
    )

# ------------------------------------------------------------
# 6. Top peer ASNs by MED diversity
# ------------------------------------------------------------

if med_non_null > 0:

    print()
    print("Top 10 peer ASNs by MED diversity:")
    print()

    print(
        med_by_peer_asn
        .sort_values(
            "unique_med_values",
            ascending=False
        )
        .head(10)
        .to_string()
    )

# ------------------------------------------------------------
# 7. MED distribution by peer
# ------------------------------------------------------------

if med_non_null > 0:

    print()
    print("Top 10 peers by MED entry count:")
    print()

    med_by_peer = (
        df.dropna(subset=["med"])
        .groupby("peer_ip")["med"]
        .agg(
            entries="count",
            unique_med_values="nunique",
            average="mean",
            minimum="min",
            maximum="max"
        )
        .sort_values(
            "entries",
            ascending=False
        )
    )

    print(
        med_by_peer
        .head(10)
        .to_string()
    )

# ------------------------------------------------------------
# 8. Prefix MED diversity
# ------------------------------------------------------------

if med_non_null > 0:

    print()
    print("MED diversity by prefix:")
    print()

    med_by_prefix = (
        df.dropna(subset=["med"])
        .groupby("prefix")["med"]
        .agg(
            entries="count",
            unique_med_values="nunique",
            minimum="min",
            maximum="max"
        )
    )

    print(
        med_by_prefix
        .sort_values(
            "unique_med_values",
            ascending=False
        )
        .head(10)
        .to_string()
    )

# ------------------------------------------------------------
# 9. MED vs AS-path length
# ------------------------------------------------------------

if med_non_null > 0:

    print()
    print("MED relationship with AS-path length:")
    print()

    med_path_analysis = (
        df.dropna(subset=["med"])
        .groupby("as_path_length")["med"]
        .agg(
            entries="count",
            average_med="mean",
            minimum_med="min",
            maximum_med="max"
        )
    )

    print(
        med_path_analysis.to_string()
    )

# ------------------------------------------------------------
# 10. MED vs ORIGIN
# ------------------------------------------------------------

if med_non_null > 0:

    print()
    print("MED statistics by ORIGIN:")
    print()

    med_origin_analysis = (
        df.dropna(subset=["med"])
        .groupby("origin")["med"]
        .agg(
            entries="count",
            unique_med_values="nunique",
            average="mean",
            minimum="min",
            maximum="max"
        )
    )

    print(
        med_origin_analysis.to_string()
    )

# ------------------------------------------------------------
# 11. High MED values
# ------------------------------------------------------------

if med_non_null > 0:

    print()
    print("Top 10 highest MED values:")
    print()

    high_med = (
        df.dropna(subset=["med"])
        .sort_values(
            "med",
            ascending=False
        )
        [
            [
                "prefix",
                "peer_ip",
                "peer_asn",
                "med",
                "origin",
                "as_path",
                "as_path_length"
            ]
        ]
        .head(10)
    )

    print(
        high_med.to_string(index=False)
    )

# ------------------------------------------------------------
# 12. Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("MED PROFILING SUMMARY")
print("=" * 60)

print(
    f"Total BGP entries       : "
    f"{total_entries:,}"
)

print(
    f"MED populated           : "
    f"{med_non_null:,}"
)

print(
    f"MED missing             : "
    f"{med_null:,}"
)

print(
    f"Unique MED values       : "
    f"{unique_med_values:,}"
)

if med_non_null > 0:
    print(
        f"Minimum MED            : "
        f"{df['med'].min():,}"
    )

    print(
        f"Maximum MED            : "
        f"{df['med'].max():,}"
    )

print()
print(
    "No causal relationship or route-selection outcome "
    "is inferred from MED values alone."
)

print(
    "MED profiling completed."
)

MED PROFILING

MED availability:

Total BGP entries : 100,000
MED populated     : 100,000 (100.00%)
MED missing       : 0 (0.00%)

MED statistics:

count    100000.000000
mean         40.599750
std         456.781758
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max       18811.000000

Most common MED values:

med
0        92272
150       2303
50        2299
1000      2291
100        684
18811       49
1430        32
5645        31
1730         6
3219         6
1640         5
3271         4
3229         3
1            3
3060         3
3279         2
2980         2
2910         1
2906         1
3201         1

Unique MED values : 22

MED statistics by peer ASN:

          entries  unique_med_values      average  minimum  maximum
peer_asn                                                           
58057        4659                  1     0.000000        0        0
34927        4588                  1     0.000000        0        0
55720       

## 2.12.4.1 MED Pattern / Peer Attribution Analysis

This section examines how MED values are distributed across peers,
peer ASNs, and prefixes.

The analysis focuses on identifying MED patterns that are strongly
associated with particular route sources.

We will examine:

- Non-zero MED distribution
- MED patterns by peer ASN
- MED patterns by peer IP
- MED diversity by peer ASN
- MED diversity by prefix
- MED values strongly associated with individual peers
- Prefixes observed with multiple MED values

The objective is to distinguish normal peer-specific MED behavior from
patterns that may warrant further investigation.

These results describe observable associations only and do not infer
causality or actual BGP route-selection outcomes.

In [52]:
# ============================================================
# 2.12.4.1 MED PATTERN / PEER ATTRIBUTION ANALYSIS
# ============================================================

print("=" * 60)
print("MED PATTERN / PEER ATTRIBUTION ANALYSIS")
print("=" * 60)

total_entries = len(df)

# ------------------------------------------------------------
# 1. Zero vs non-zero MED
# ------------------------------------------------------------

zero_med = (df["med"] == 0).sum()
nonzero_med = (df["med"] != 0).sum()

print()
print("MED zero / non-zero distribution:")
print()

print(f"MED = 0     : {zero_med:,} "
      f"({zero_med / total_entries * 100:.2f}%)")

print(f"MED != 0    : {nonzero_med:,} "
      f"({nonzero_med / total_entries * 100:.2f}%)")

# ------------------------------------------------------------
# 2. Non-zero MED distribution
# ------------------------------------------------------------

print()
print("Most common non-zero MED values:")
print()

nonzero_med_distribution = (
    df[df["med"] != 0]["med"]
    .value_counts()
    .head(20)
)

print(
    nonzero_med_distribution.to_string()
)

# ------------------------------------------------------------
# 3. MED pattern by peer ASN
# ------------------------------------------------------------

peer_asn_med = (
    df.groupby("peer_asn")["med"]
    .agg(
        entries="size",
        unique_med_values="nunique",
        nonzero_med_entries=lambda x: (x != 0).sum(),
        minimum="min",
        maximum="max"
    )
)

peer_asn_med["nonzero_med_pct"] = (
    peer_asn_med["nonzero_med_entries"]
    / peer_asn_med["entries"]
    * 100
)

print()
print("Top peer ASNs by non-zero MED count:")
print()

print(
    peer_asn_med
    .sort_values(
        "nonzero_med_entries",
        ascending=False
    )
    .head(10)
    .to_string()
)

# ------------------------------------------------------------
# 4. Peer ASNs by non-zero MED percentage
# ------------------------------------------------------------

print()
print("Top peer ASNs by non-zero MED percentage:")
print()

print(
    peer_asn_med[
        peer_asn_med["entries"] >= 10
    ]
    .sort_values(
        "nonzero_med_pct",
        ascending=False
    )
    .head(10)
    .to_string()
)

# ------------------------------------------------------------
# 5. MED diversity by prefix
# ------------------------------------------------------------

prefix_med = (
    df.groupby("prefix")["med"]
    .agg(
        entries="size",
        unique_med_values="nunique",
        minimum="min",
        maximum="max"
    )
)

multi_med_prefixes = prefix_med[
    prefix_med["unique_med_values"] > 1
].copy()

print()
print("MED diversity by prefix:")
print()

print(
    f"Total prefixes                 : "
    f"{len(prefix_med):,}"
)

print(
    f"Prefixes with multiple MEDs    : "
    f"{len(multi_med_prefixes):,}"
)

print(
    f"Percentage with multiple MEDs  : "
    f"{len(multi_med_prefixes) / len(prefix_med) * 100:.2f}%"
)

# ------------------------------------------------------------
# 6. Top prefixes by MED diversity
# ------------------------------------------------------------

print()
print("Top 10 prefixes by MED diversity:")
print()

print(
    multi_med_prefixes
    .sort_values(
        ["unique_med_values", "entries"],
        ascending=False
    )
    .head(10)
    .to_string()
)

# ------------------------------------------------------------
# 7. MED value -> peer ASN relationship
# ------------------------------------------------------------

print()
print("Non-zero MED values and associated peer ASNs:")
print()

med_peer_relationship = (
    df[df["med"] != 0]
    .groupby("med")
    .agg(
        entries=("med", "size"),
        unique_peers=("peer_ip", "nunique"),
        unique_peer_asns=("peer_asn", "nunique")
    )
    .sort_values(
        "entries",
        ascending=False
    )
)

print(
    med_peer_relationship
    .head(20)
    .to_string()
)

# ------------------------------------------------------------
# 8. MED values concentrated in one peer ASN
# ------------------------------------------------------------

print()
print("MED values with strongest peer-ASN concentration:")
print()

med_asn_counts = (
    df[df["med"] != 0]
    .groupby(["med", "peer_asn"])
    .size()
    .reset_index(name="entries")
)

top_peer_for_med = (
    med_asn_counts
    .sort_values(
        ["med", "entries"],
        ascending=[True, False]
    )
    .groupby("med")
    .head(1)
)

med_totals = (
    df[df["med"] != 0]
    .groupby("med")
    .size()
    .rename("total_med_entries")
)

top_peer_for_med = top_peer_for_med.merge(
    med_totals,
    on="med"
)

top_peer_for_med["peer_share_pct"] = (
    top_peer_for_med["entries"]
    / top_peer_for_med["total_med_entries"]
    * 100
)

print(
    top_peer_for_med
    .sort_values(
        "peer_share_pct",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 9. Highest MED observations
# ------------------------------------------------------------

print()
print("Top 10 highest MED observations:")
print()

highest_med = (
    df[
        [
            "prefix",
            "peer_ip",
            "peer_asn",
            "med",
            "origin",
            "as_path",
            "as_path_length"
        ]
    ]
    .sort_values(
        "med",
        ascending=False
    )
    .head(10)
)

print(
    highest_med.to_string(index=False)
)

# ------------------------------------------------------------
# 10. Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("MED PATTERN / PEER ATTRIBUTION SUMMARY")
print("=" * 60)

print(
    f"Total BGP entries              : "
    f"{total_entries:,}"
)

print(
    f"MED = 0 entries                : "
    f"{zero_med:,}"
)

print(
    f"MED != 0 entries               : "
    f"{nonzero_med:,}"
)

print(
    f"Unique MED values              : "
    f"{df['med'].nunique():,}"
)

print(
    f"Prefixes with multiple MEDs    : "
    f"{len(multi_med_prefixes):,}"
)

print()
print(
    "No causal relationship or route-selection outcome "
    "is inferred from MED patterns."
)

print(
    "MED pattern / peer attribution analysis completed."
)

MED PATTERN / PEER ATTRIBUTION ANALYSIS

MED zero / non-zero distribution:

MED = 0     : 92,272 (92.27%)
MED != 0    : 7,728 (7.73%)

Most common non-zero MED values:

med
150      2303
50       2299
1000     2291
100       684
18811      49
1430       32
5645       31
1730        6
3219        6
1640        5
3271        4
3229        3
1           3
3060        3
3279        2
2980        2
2910        1
2906        1
3201        1
3323        1

Top peer ASNs by non-zero MED count:

          entries  unique_med_values  nonzero_med_entries  minimum  maximum  nonzero_med_pct
peer_asn                                                                                    
204092       2303                  1                 2303      150      150       100.000000
29504        2299                  1                 2299       50       50       100.000000
49432        2291                  1                 2291     1000     1000       100.000000
50628         685                  2       

## 2.12.4.2 MED Comparison-Set Profiling

MED is primarily meaningful when comparing multiple routes to the same
destination learned from the same neighboring AS.

This section identifies MED comparison sets by grouping route candidates
using:

    prefix + peer ASN

Within each comparison set, we examine:

- Number of route candidates
- Number of unique MED values
- Minimum MED
- Maximum MED
- MED range
- Whether multiple MED values are present
- Prefixes with the greatest MED variation

The objective is to identify genuine MED comparison sets rather than
simply comparing MED values across unrelated neighboring ASNs.

A lower MED may be preferred when MED is compared between applicable
routes from the same neighboring AS, subject to the router's BGP
configuration and the rest of the best-path decision process.

This analysis identifies possible MED comparison opportunities only.
It does not determine the actual BGP best path or infer causality.

In [53]:
# ============================================================
# 2.12.4.2 MED COMPARISON-SET PROFILING
# ============================================================

print("=" * 60)
print("MED COMPARISON-SET PROFILING")
print("=" * 60)

total_entries = len(df)

# ------------------------------------------------------------
# 1. Build MED comparison sets
#
# Same prefix + same peer ASN
# ------------------------------------------------------------

med_comparison_sets = (
    df.groupby(
        ["prefix", "peer_asn"]
    )["med"]
    .agg(
        route_candidates="size",
        unique_med_values="nunique",
        minimum_med="min",
        maximum_med="max"
    )
    .reset_index()
)

med_comparison_sets["med_range"] = (
    med_comparison_sets["maximum_med"]
    - med_comparison_sets["minimum_med"]
)

# ------------------------------------------------------------
# 2. Identify sets with multiple MED values
# ------------------------------------------------------------

multi_med_sets = med_comparison_sets[
    med_comparison_sets["unique_med_values"] > 1
].copy()

print()
print("MED comparison-set inventory:")
print()

print(
    f"Total comparison sets          : "
    f"{len(med_comparison_sets):,}"
)

print(
    f"Sets with multiple MED values  : "
    f"{len(multi_med_sets):,}"
)

print(
    f"Percentage with multiple MEDs  : "
    f"{len(multi_med_sets) / len(med_comparison_sets) * 100:.2f}%"
)

# ------------------------------------------------------------
# 3. Comparison sets with multiple route candidates
# ------------------------------------------------------------

multi_route_sets = med_comparison_sets[
    med_comparison_sets["route_candidates"] > 1
].copy()

print()
print("Comparison sets with multiple route candidates:")
print()

print(
    f"Sets with multiple candidates  : "
    f"{len(multi_route_sets):,}"
)

print(
    f"Sets with multiple MED values  : "
    f"{len(multi_med_sets):,}"
)

# ------------------------------------------------------------
# 4. Top comparison sets by MED range
# ------------------------------------------------------------

print()
print("Top 10 MED comparison sets by MED range:")
print()

top_med_ranges = (
    multi_med_sets
    .sort_values(
        ["med_range", "unique_med_values"],
        ascending=False
    )
    .head(10)
)

print(
    top_med_ranges.to_string(index=False)
)

# ------------------------------------------------------------
# 5. Top comparison sets by MED diversity
# ------------------------------------------------------------

print()
print("Top 10 MED comparison sets by MED diversity:")
print()

top_med_diversity = (
    multi_med_sets
    .sort_values(
        ["unique_med_values", "route_candidates"],
        ascending=False
    )
    .head(10)
)

print(
    top_med_diversity.to_string(index=False)
)

# ------------------------------------------------------------
# 6. Detailed route candidates for top MED ranges
# ------------------------------------------------------------

print()
print("Detailed routes for top MED comparison sets:")
print()

if len(top_med_ranges) > 0:

    top_pairs = list(
        zip(
            top_med_ranges["prefix"],
            top_med_ranges["peer_asn"]
        )
    )

    detail_mask = pd.Series(False, index=df.index)

    for prefix_value, peer_asn_value in top_pairs:

        detail_mask = (
            detail_mask
            |
            (
                (df["prefix"] == prefix_value)
                &
                (df["peer_asn"] == peer_asn_value)
            )
        )

    comparison_detail = (
        df.loc[
            detail_mask,
            [
                "prefix",
                "peer_ip",
                "peer_asn",
                "med",
                "origin",
                "as_path",
                "as_path_length"
            ]
        ]
        .sort_values(
            ["prefix", "peer_asn", "med"]
        )
    )

    print(
        comparison_detail.to_string(index=False)
    )

else:

    print(
        "No comparison sets with multiple MED values were found."
    )

# ------------------------------------------------------------
# 7. MED comparison direction
# ------------------------------------------------------------

print()
print("MED comparison characteristics:")
print()

if len(multi_med_sets) > 0:

    med_comparison_stats = (
        multi_med_sets[
            [
                "minimum_med",
                "maximum_med",
                "med_range",
                "unique_med_values"
            ]
        ]
        .describe()
    )

    print(
        med_comparison_stats.to_string()
    )

else:

    print(
        "No multi-MED comparison sets available."
    )

# ------------------------------------------------------------
# 8. Prefixes with the greatest MED comparison variation
# ------------------------------------------------------------

print()
print("Top 10 prefixes by MED comparison variation:")
print()

if len(multi_med_sets) > 0:

    prefix_variation = (
        multi_med_sets
        .groupby("prefix")
        .agg(
            comparison_sets=("peer_asn", "size"),
            peer_asns=("peer_asn", "nunique"),
            max_med_range=("med_range", "max"),
            max_unique_med_values=(
                "unique_med_values",
                "max"
            )
        )
        .sort_values(
            ["max_med_range", "max_unique_med_values"],
            ascending=False
        )
        .head(10)
    )

    print(
        prefix_variation.to_string()
    )

# ------------------------------------------------------------
# 9. Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("MED COMPARISON-SET PROFILING SUMMARY")
print("=" * 60)

print(
    f"Total BGP entries             : "
    f"{total_entries:,}"
)

print(
    f"Total prefix / peer-ASN sets  : "
    f"{len(med_comparison_sets):,}"
)

print(
    f"Sets with multiple MED values : "
    f"{len(multi_med_sets):,}"
)

if len(multi_med_sets) > 0:

    print(
        f"Maximum MED range             : "
        f"{multi_med_sets['med_range'].max():,}"
    )

    print(
        f"Maximum MED diversity         : "
        f"{multi_med_sets['unique_med_values'].max():,}"
    )

print()
print(
    "No actual BGP best-path decision is inferred."
)

print(
    "MED comparison-set profiling completed."
)

MED COMPARISON-SET PROFILING

MED comparison-set inventory:

Total comparison sets          : 92,501
Sets with multiple MED values  : 0
Percentage with multiple MEDs  : 0.00%

Comparison sets with multiple route candidates:

Sets with multiple candidates  : 7,439
Sets with multiple MED values  : 0

Top 10 MED comparison sets by MED range:

Empty DataFrame
Columns: [prefix, peer_asn, route_candidates, unique_med_values, minimum_med, maximum_med, med_range]
Index: []

Top 10 MED comparison sets by MED diversity:

Empty DataFrame
Columns: [prefix, peer_asn, route_candidates, unique_med_values, minimum_med, maximum_med, med_range]
Index: []

Detailed routes for top MED comparison sets:

No comparison sets with multiple MED values were found.

MED comparison characteristics:

No multi-MED comparison sets available.

Top 10 prefixes by MED comparison variation:


MED COMPARISON-SET PROFILING SUMMARY
Total BGP entries             : 100,000
Total prefix / peer-ASN sets  : 92,501
Sets with mult

## 2.12.4.3 MED / AS-Path Relationship Analysis

This section examines observable relationships between MED and AS-path
structure.

We will compare MED against:

- AS-path length
- Number of unique ASNs
- AS-path repetition
- Maximum consecutive ASN repetition / prepending depth
- Long AS-path classification
- Extreme-prepending classification

The objective is to determine whether particular MED patterns are
associated with specific AS-path structures.

This is an observational analysis only. No causal relationship between
MED and AS-path characteristics is inferred.

In [54]:
# ============================================================
# 2.12.4.3 MED / AS-PATH RELATIONSHIP ANALYSIS
# ============================================================

print("=" * 60)
print("MED / AS-PATH RELATIONSHIP ANALYSIS")
print("=" * 60)

total_entries = len(df)

# ------------------------------------------------------------
# 1. Prepare analysis columns
# ------------------------------------------------------------

# Unique ASN count
if "unique_asn_count" not in df.columns:
    df["unique_asn_count"] = (
        df["as_path_list"]
        .apply(lambda x: len(set(x)) if x else 0)
    )

# Repetition count
if "repetition_count" not in df.columns:
    df["repetition_count"] = (
        df["as_path_length"]
        - df["unique_asn_count"]
    )

# Extreme prepending
EXTREME_PREPEND_THRESHOLD = 5

df["_is_extreme_prepend"] = (
    df["max_consecutive_repeat"]
    >= EXTREME_PREPEND_THRESHOLD
)

# Long AS-path
LONG_PATH_THRESHOLD = 10

df["_is_long_path"] = (
    df["as_path_length"]
    >= LONG_PATH_THRESHOLD
)

# ------------------------------------------------------------
# 2. MED statistics by AS-path length
# ------------------------------------------------------------

print()
print("MED statistics by AS-path length:")
print()

med_by_path_length = (
    df.groupby("as_path_length")["med"]
    .agg(
        entries="size",
        average_med="mean",
        median_med="median",
        minimum_med="min",
        maximum_med="max",
        nonzero_med_entries=lambda x: (x != 0).sum()
    )
)

med_by_path_length["nonzero_med_pct"] = (
    med_by_path_length["nonzero_med_entries"]
    / med_by_path_length["entries"]
    * 100
)

print(
    med_by_path_length.to_string()
)

# ------------------------------------------------------------
# 3. MED statistics by unique ASN count
# ------------------------------------------------------------

print()
print("MED statistics by unique ASN count:")
print()

med_by_unique_asns = (
    df.groupby("unique_asn_count")["med"]
    .agg(
        entries="size",
        average_med="mean",
        median_med="median",
        minimum_med="min",
        maximum_med="max",
        nonzero_med_entries=lambda x: (x != 0).sum()
    )
)

med_by_unique_asns["nonzero_med_pct"] = (
    med_by_unique_asns["nonzero_med_entries"]
    / med_by_unique_asns["entries"]
    * 100
)

print(
    med_by_unique_asns.to_string()
)

# ------------------------------------------------------------
# 4. MED statistics by repetition count
# ------------------------------------------------------------

print()
print("MED statistics by AS-path repetition count:")
print()

med_by_repetition = (
    df.groupby("repetition_count")["med"]
    .agg(
        entries="size",
        average_med="mean",
        median_med="median",
        minimum_med="min",
        maximum_med="max",
        nonzero_med_entries=lambda x: (x != 0).sum()
    )
)

med_by_repetition["nonzero_med_pct"] = (
    med_by_repetition["nonzero_med_entries"]
    / med_by_repetition["entries"]
    * 100
)

print(
    med_by_repetition.to_string()
)

# ------------------------------------------------------------
# 5. MED statistics by prepend depth
# ------------------------------------------------------------

print()
print("MED statistics by maximum prepend depth:")
print()

med_by_prepend_depth = (
    df.groupby("max_consecutive_repeat")["med"]
    .agg(
        entries="size",
        average_med="mean",
        median_med="median",
        minimum_med="min",
        maximum_med="max",
        nonzero_med_entries=lambda x: (x != 0).sum()
    )
)

med_by_prepend_depth["nonzero_med_pct"] = (
    med_by_prepend_depth["nonzero_med_entries"]
    / med_by_prepend_depth["entries"]
    * 100
)

print(
    med_by_prepend_depth.to_string()
)

# ------------------------------------------------------------
# 6. Long path vs normal path
# ------------------------------------------------------------

print()
print("MED distribution: long vs normal AS-paths:")
print()

path_class_summary = (
    df.assign(
        path_class=df["_is_long_path"].map(
            {
                True: "Long path (>=10)",
                False: "Normal path (<10)"
            }
        )
    )
    .groupby("path_class")["med"]
    .agg(
        entries="size",
        average_med="mean",
        median_med="median",
        minimum_med="min",
        maximum_med="max",
        nonzero_med_entries=lambda x: (x != 0).sum()
    )
)

path_class_summary["nonzero_med_pct"] = (
    path_class_summary["nonzero_med_entries"]
    / path_class_summary["entries"]
    * 100
)

print(
    path_class_summary.to_string()
)

# ------------------------------------------------------------
# 7. Extreme prepending vs normal
# ------------------------------------------------------------

print()
print("MED distribution: extreme prepending vs non-extreme:")
print()

prepend_class_summary = (
    df.assign(
        prepend_class=df["_is_extreme_prepend"].map(
            {
                True: "Extreme prepend (>=5)",
                False: "Non-extreme prepend"
            }
        )
    )
    .groupby("prepend_class")["med"]
    .agg(
        entries="size",
        average_med="mean",
        median_med="median",
        minimum_med="min",
        maximum_med="max",
        nonzero_med_entries=lambda x: (x != 0).sum()
    )
)

prepend_class_summary["nonzero_med_pct"] = (
    prepend_class_summary["nonzero_med_entries"]
    / prepend_class_summary["entries"]
    * 100
)

print(
    prepend_class_summary.to_string()
)

# ------------------------------------------------------------
# 8. Most common MED values among long paths
# ------------------------------------------------------------

print()
print("Most common MED values among long AS-paths:")
print()

long_path_med = (
    df.loc[df["_is_long_path"], "med"]
    .value_counts()
    .head(15)
)

print(
    long_path_med.to_string()
)

# ------------------------------------------------------------
# 9. Most common MED values among extreme prepending
# ------------------------------------------------------------

print()
print("Most common MED values among extreme-prepending paths:")
print()

extreme_med = (
    df.loc[df["_is_extreme_prepend"], "med"]
    .value_counts()
    .head(15)
)

print(
    extreme_med.to_string()
)

# ------------------------------------------------------------
# 10. High MED observations with AS-path structure
# ------------------------------------------------------------

print()
print("Top 10 highest MED observations with AS-path structure:")
print()

high_med_view = (
    df[
        [
            "prefix",
            "peer_ip",
            "peer_asn",
            "med",
            "as_path_length",
            "unique_asn_count",
            "repetition_count",
            "max_consecutive_repeat",
            "origin"
        ]
    ]
    .sort_values("med", ascending=False)
    .head(10)
)

print(
    high_med_view.to_string(index=False)
)

# ------------------------------------------------------------
# 11. Non-zero MED vs zero MED structural comparison
# ------------------------------------------------------------

print()
print("AS-path structure: MED = 0 vs MED != 0:")
print()

med_class_summary = (
    df.assign(
        med_class=df["med"].map(
            {
                0: "MED = 0",
            }
        )
    )
)

med_class_summary["med_class"] = (
    med_class_summary["med"]
    .apply(
        lambda x: "MED = 0"
        if x == 0
        else "MED != 0"
    )
)

structural_by_med = (
    med_class_summary
    .groupby("med_class")
    .agg(
        entries=("med", "size"),
        avg_path_length=("as_path_length", "mean"),
        avg_unique_asns=("unique_asn_count", "mean"),
        avg_repetition=("repetition_count", "mean"),
        avg_prepend_depth=(
            "max_consecutive_repeat",
            "mean"
        ),
        long_paths=("_is_long_path", "sum"),
        extreme_prepending=(
            "_is_extreme_prepend",
            "sum"
        )
    )
)

structural_by_med["long_path_pct"] = (
    structural_by_med["long_paths"]
    / structural_by_med["entries"]
    * 100
)

structural_by_med["extreme_prepend_pct"] = (
    structural_by_med["extreme_prepending"]
    / structural_by_med["entries"]
    * 100
)

print(
    structural_by_med.to_string()
)

# ------------------------------------------------------------
# 12. Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("MED / AS-PATH RELATIONSHIP SUMMARY")
print("=" * 60)

print(
    f"Total BGP entries              : "
    f"{total_entries:,}"
)

print(
    f"MED values analyzed            : "
    f"{df['med'].nunique():,}"
)

print(
    f"Long-path threshold            : "
    f">= {LONG_PATH_THRESHOLD}"
)

print(
    f"Extreme-prepend threshold      : "
    f">= {EXTREME_PREPEND_THRESHOLD}"
)

print()
print(
    "The analysis identifies observable associations "
    "between MED and AS-path structure."
)

print(
    "No causal relationship is inferred."
)

# ------------------------------------------------------------
# Cleanup temporary columns
# ------------------------------------------------------------

df.drop(
    columns=[
        "_is_extreme_prepend",
        "_is_long_path"
    ],
    inplace=True
)

print()
print(
    "MED / AS-path relationship analysis completed."
)

MED / AS-PATH RELATIONSHIP ANALYSIS

MED statistics by AS-path length:

                entries  average_med  median_med  minimum_med  maximum_med  nonzero_med_entries  nonzero_med_pct
as_path_length                                                                                                  
1                   102     0.029412         0.0            0            1                    3         2.941176
2                  5550    12.144144         0.0            0         3271                  344         6.198198
3                 27792    46.442717         0.0            0        18811                  934         3.360679
4                 31783    39.691722         0.0            0         5645                 3056         9.615203
5                 17438    49.043239         0.0            0         5645                 1843        10.568873
6                  9152    30.763767         0.0            0         1000                  837         9.145542
7                  4708 

## 2.12.5 Local Preference Profiling

This section profiles the BGP LOCAL_PREF attribute as a
control-plane routing-policy attribute.

LOCAL_PREF is commonly used within an autonomous system to
influence outbound route selection. A higher LOCAL_PREF value
is normally preferred over a lower value, subject to the
router's BGP decision process and configuration.

This analysis examines:

- LOCAL_PREF availability
- LOCAL_PREF value distribution
- LOCAL_PREF diversity
- LOCAL_PREF distribution by peer ASN
- LOCAL_PREF distribution by prefix
- Prefixes with multiple LOCAL_PREF values
- Relationship between LOCAL_PREF and AS-path structure
- Potentially unusual LOCAL_PREF patterns

The objective is to understand the routing-policy structure
represented in the BGP dataset.

This dataset does not contain traffic-volume contribution
per prefix. Therefore, this analysis cannot determine the
actual outbound traffic-engineering impact of LOCAL_PREF.

The analysis describes control-plane routing-policy patterns
only and does not infer actual traffic volume or causality.

In [55]:
# ============================================================
# 2.12.5 LOCAL PREFERENCE PROFILING
# ============================================================

print("=" * 60)
print("LOCAL PREFERENCE PROFILING")
print("=" * 60)

total_entries = len(df)

# ------------------------------------------------------------
# 1. LOCAL_PREF availability
# ------------------------------------------------------------

local_pref_non_null = df["local_pref"].notna().sum()
local_pref_null = df["local_pref"].isna().sum()

print()
print("LOCAL_PREF availability:")
print()

print(
    f"Total BGP entries : "
    f"{total_entries:,}"
)

print(
    f"LOCAL_PREF populated : "
    f"{local_pref_non_null:,} "
    f"({local_pref_non_null / total_entries * 100:.2f}%)"
)

print(
    f"LOCAL_PREF missing   : "
    f"{local_pref_null:,} "
    f"({local_pref_null / total_entries * 100:.2f}%)"
)

# ------------------------------------------------------------
# 2. LOCAL_PREF statistics
# ------------------------------------------------------------

if local_pref_non_null > 0:

    print()
    print("LOCAL_PREF statistics:")
    print()

    local_pref_stats = (
        df["local_pref"]
        .describe()
    )

    print(
        local_pref_stats.to_string()
    )

# ------------------------------------------------------------
# 3. LOCAL_PREF value distribution
# ------------------------------------------------------------

if local_pref_non_null > 0:

    print()
    print("Most common LOCAL_PREF values:")
    print()

    local_pref_distribution = (
        df["local_pref"]
        .value_counts(dropna=True)
        .head(20)
    )

    print(
        local_pref_distribution.to_string()
    )

# ------------------------------------------------------------
# 4. LOCAL_PREF diversity
# ------------------------------------------------------------

unique_local_pref_values = (
    df["local_pref"]
    .dropna()
    .nunique()
)

print()
print(
    f"Unique LOCAL_PREF values : "
    f"{unique_local_pref_values:,}"
)

# ------------------------------------------------------------
# 5. LOCAL_PREF distribution by peer ASN
# ------------------------------------------------------------

if local_pref_non_null > 0:

    print()
    print("LOCAL_PREF statistics by peer ASN:")
    print()

    local_pref_by_peer_asn = (
        df.dropna(subset=["local_pref"])
        .groupby("peer_asn")["local_pref"]
        .agg(
            entries="count",
            unique_values="nunique",
            average="mean",
            minimum="min",
            maximum="max"
        )
        .sort_values(
            "entries",
            ascending=False
        )
    )

    print(
        local_pref_by_peer_asn.to_string()
    )

# ------------------------------------------------------------
# 6. Peer ASNs by LOCAL_PREF diversity
# ------------------------------------------------------------

if local_pref_non_null > 0:

    print()
    print("Top 10 peer ASNs by LOCAL_PREF diversity:")
    print()

    print(
        local_pref_by_peer_asn
        .sort_values(
            "unique_values",
            ascending=False
        )
        .head(10)
        .to_string()
    )

# ------------------------------------------------------------
# 7. LOCAL_PREF distribution by prefix
# ------------------------------------------------------------

if local_pref_non_null > 0:

    local_pref_by_prefix = (
        df.dropna(subset=["local_pref"])
        .groupby("prefix")["local_pref"]
        .agg(
            entries="count",
            unique_values="nunique",
            minimum="min",
            maximum="max"
        )
    )

    multi_local_pref_prefixes = (
        local_pref_by_prefix[
            local_pref_by_prefix["unique_values"] > 1
        ]
        .copy()
    )

    print()
    print("LOCAL_PREF diversity by prefix:")
    print()

    print(
        f"Total prefixes              : "
        f"{len(local_pref_by_prefix):,}"
    )

    print(
        f"Prefixes with multiple values: "
        f"{len(multi_local_pref_prefixes):,}"
    )

    print(
        f"Percentage                   : "
        f"{len(multi_local_pref_prefixes) / len(local_pref_by_prefix) * 100:.2f}%"
    )

# ------------------------------------------------------------
# 8. Top prefixes by LOCAL_PREF diversity
# ------------------------------------------------------------

if local_pref_non_null > 0:

    print()
    print("Top 10 prefixes by LOCAL_PREF diversity:")
    print()

    print(
        multi_local_pref_prefixes
        .sort_values(
            ["unique_values", "entries"],
            ascending=False
        )
        .head(10)
        .to_string()
    )

# ------------------------------------------------------------
# 9. LOCAL_PREF distribution by ORIGIN
# ------------------------------------------------------------

if local_pref_non_null > 0:

    print()
    print("LOCAL_PREF statistics by ORIGIN:")
    print()

    local_pref_by_origin = (
        df.dropna(subset=["local_pref"])
        .groupby("origin")["local_pref"]
        .agg(
            entries="count",
            unique_values="nunique",
            average="mean",
            minimum="min",
            maximum="max"
        )
    )

    print(
        local_pref_by_origin.to_string()
    )

# ------------------------------------------------------------
# 10. LOCAL_PREF relationship with AS-path length
# ------------------------------------------------------------

if local_pref_non_null > 0:

    print()
    print("LOCAL_PREF statistics by AS-path length:")
    print()

    local_pref_by_path_length = (
        df.dropna(subset=["local_pref"])
        .groupby("as_path_length")["local_pref"]
        .agg(
            entries="count",
            average="mean",
            median="median",
            minimum="min",
            maximum="max"
        )
    )

    print(
        local_pref_by_path_length.to_string()
    )

# ------------------------------------------------------------
# 11. Highest LOCAL_PREF observations
# ------------------------------------------------------------

if local_pref_non_null > 0:

    print()
    print("Top 10 highest LOCAL_PREF observations:")
    print()

    highest_local_pref = (
        df[
            [
                "prefix",
                "peer_ip",
                "peer_asn",
                "local_pref",
                "origin",
                "as_path",
                "as_path_length"
            ]
        ]
        .sort_values(
            "local_pref",
            ascending=False
        )
        .head(10)
    )

    print(
        highest_local_pref.to_string(index=False)
    )

# ------------------------------------------------------------
# 12. Lowest LOCAL_PREF observations
# ------------------------------------------------------------

if local_pref_non_null > 0:

    print()
    print("Top 10 lowest LOCAL_PREF observations:")
    print()

    lowest_local_pref = (
        df[
            [
                "prefix",
                "peer_ip",
                "peer_asn",
                "local_pref",
                "origin",
                "as_path",
                "as_path_length"
            ]
        ]
        .sort_values(
            "local_pref",
            ascending=True
        )
        .head(10)
    )

    print(
        lowest_local_pref.to_string(index=False)
    )

# ------------------------------------------------------------
# 13. Summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("LOCAL PREFERENCE PROFILING SUMMARY")
print("=" * 60)

print(
    f"Total BGP entries          : "
    f"{total_entries:,}"
)

print(
    f"LOCAL_PREF populated       : "
    f"{local_pref_non_null:,}"
)

print(
    f"LOCAL_PREF missing         : "
    f"{local_pref_null:,}"
)

print(
    f"Unique LOCAL_PREF values   : "
    f"{unique_local_pref_values:,}"
)

if local_pref_non_null > 0:

    print(
        f"Minimum LOCAL_PREF         : "
        f"{df['local_pref'].min():,}"
    )

    print(
        f"Maximum LOCAL_PREF         : "
        f"{df['local_pref'].max():,}"
    )

if local_pref_non_null > 0:

    print(
        f"Prefixes with multiple "
        f"LOCAL_PREF values        : "
        f"{len(multi_local_pref_prefixes):,}"
    )

print()
print(
    "LOCAL_PREF profiling describes control-plane "
    "routing-policy patterns."
)

print(
    "Actual traffic-engineering impact cannot be "
    "determined without traffic-volume data."
)

print(
    "No causal relationship is inferred."
)

print(
    "LOCAL_PREF profiling completed."
)

LOCAL PREFERENCE PROFILING

LOCAL_PREF availability:

Total BGP entries : 100,000
LOCAL_PREF populated : 100,000 (100.00%)
LOCAL_PREF missing   : 0 (0.00%)

LOCAL_PREF statistics:

count    100000.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0

Most common LOCAL_PREF values:

local_pref
0    100000

Unique LOCAL_PREF values : 1

LOCAL_PREF statistics by peer ASN:

          entries  unique_values  average  minimum  maximum
peer_asn                                                   
58057        4659              1      0.0        0        0
34927        4588              1      0.0        0        0
55720        4586              1      0.0        0        0
34800        2621              1      0.0        0        0
202365       2548              1      0.0        0        0
44393        2307              1      0.0        0        0
37721        2305              1      0.0        0        0
204092      